
# Análise Exploratória de Dados — E-commerce Brasileiro (Olist)

**Projeto:** Técnicas de Programação I — Python  
**Curso:** CAIXAVERSO | Analista de Dados II  

**Instrutora:** Stephanie Pumarino

**Aluna:** Joyce de Morais Siqueira

## 1. Contexto e objetivo

O comércio eletrônico permite que consumidores realizem compras em diferentes regiões do Brasil. A análise dos pedidos pode ajudar a compreender o comportamento das vendas, a distribuição geográfica dos clientes e as categorias de produtos comercializados.

Este projeto utiliza dados públicos do e-commerce brasileiro Olist para realizar uma Análise Exploratória de Dados (EDA), aplicando técnicas de inspeção, limpeza, transformação e visualização de dados com Python.

### Pergunta principal

Como evoluíram as vendas da Olist ao longo do período analisado e quais diferenças podem ser observadas entre estados, valores dos pedidos e categorias de produtos?

### Perguntas auxiliares

1. Como evoluiu mensalmente o valor das vendas?
2. Quais estados concentram o maior valor de vendas?
3. Qual é a distribuição dos valores dos pedidos?
4. Quais categorias de produtos apresentam maior valor de vendas?

### Objetivo

Identificar padrões temporais, geográficos e comerciais, utilizando estatísticas descritivas e visualizações para apresentar conclusões fundamentadas nos dados.

A análise terá como foco os pedidos com status de entrega concluída. O valor de venda será calculado a partir dos preços dos itens, sem incluir o frete.



## 2. Fontes e descrição dos dados

Os dados utilizados pertencem ao Brazilian E-Commerce Public Dataset by Olist, disponibilizado publicamente no Kaggle.

**Fonte:** https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

**Instituição responsável:** Olist  
**Data de acesso:** 18/11/2026  
**Período:** será confirmado por meio da inspeção das datas de compra.

### Arquivos utilizados

- `olist_orders_dataset.csv`: informações dos pedidos, datas e status.
- `olist_order_items_dataset.csv`: produtos vendidos, preços e fretes.
- `olist_products_dataset.csv`: características e categorias dos produtos.
- `olist_customers_dataset.csv`: identificação e localização dos clientes.

### Unidade de análise

Os arquivos possuem granularidades diferentes:

- Pedidos: uma linha por pedido.
- Itens: uma linha por item de um pedido.
- Produtos: uma linha por produto cadastrado.
- Clientes: uma linha por identificação de cliente associada ao pedido.

As bases serão relacionadas pelas chaves `order_id`, `product_id` e `customer_id`.

Para analisar valores por pedido, será necessário agregar os itens, evitando contar um mesmo pedido várias vezes.



## 3. Carregamento e Inspeção Inicial dos Dados

### Objetivo da etapa

Nesta etapa, será realizada a leitura dos quatro arquivos CSV da Olist, seguida de uma inspeção inicial para compreender a estrutura, o conteúdo e as características dos dados.

Antes de iniciar qualquer procedimento de limpeza ou transformação, é fundamental conhecer as bases e identificar possíveis problemas de qualidade.

### Procedimentos realizados

1. **Importação das bibliotecas:** utilização de Pandas, NumPy e Matplotlib.
2. **Carregamento dos dados:** leitura dos quatro arquivos CSV.
3. **Dimensões das bases:** identificação da quantidade de linhas e colunas com `shape`.
4. **Visualização dos registros:** utilização de `head()` e `tail()`.
5. **Estrutura dos dados:** identificação das colunas, tipos de dados e uso de memória com `columns`, `dtypes` e `info()`.
6. **Estatísticas descritivas:** utilização de `describe()` para analisar as características das variáveis.
7. **Análise inicial dos pedidos:** identificação do período das compras e da distribuição dos status.

### Resultado esperado

Ao final desta etapa, será possível compreender a estrutura dos conjuntos de dados, identificar as variáveis relevantes para responder às perguntas do projeto e reconhecer possíveis inconsistências que serão investigadas na próxima etapa.

**Importante:** nesta fase, os dados serão apenas carregados e inspecionados, sem exclusão ou alteração dos registros originais.


In [ ]:
#Importando bibliotecas

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

print("Bibliotecas importadas com sucesso!")


In [ ]:
#Leitura dos arquivos CSV

df_orders = pd.read_csv("olist_orders_dataset.csv")
df_items = pd.read_csv("olist_order_items_dataset.csv")
df_products = pd.read_csv("olist_products_dataset.csv")
df_customers = pd.read_csv("olist_customers_dataset.csv")

print("Arquivos carregados com sucesso!")


In [ ]:
#Conferindo as dimensões dos datasets

bases = {
    "Pedidos": df_orders,
    "Itens": df_items,
    "Produtos": df_products,
    "Clientes": df_customers
}

for nome, df in bases.items():
    print(f"{nome}: {df.shape[0]} linhas e {df.shape[1]} colunas")


In [ ]:
#Visualizando os registros dos datasets


for nome, df in bases.items():
    print(f"\n===== {nome.upper()} =====")
    display(df.head())
    display(df.tail(3))


In [ ]:
#Inspecionando os tipos de memória e valores nulos dos datasets

for nome, df in bases.items():
    print(f"\n===== {nome.upper()} =====")
    print("Colunas:", df.columns.tolist())
    df.info(memory_usage="deep")


In [ ]:
#Estatisticas descritivas dos datasets


for nome, df in bases.items():
    print(f"\n===== {nome.upper()} =====")
    display(df.describe(include="all").T)


In [ ]:
#Conferindo período e status dos pedidos


datas_compra = pd.to_datetime(
    df_orders["order_purchase_timestamp"],
    errors="coerce"
)

print("Primeira compra:", datas_compra.min())
print("Última compra:", datas_compra.max())

print("\nStatus dos pedidos:")
display(df_orders["order_status"].value_counts(dropna=False))



## 4. Diagnóstico e Limpeza dos Dados

### Objetivo da etapa

Esta etapa tem como objetivo avaliar a qualidade dos dados da Olist, identificando possíveis problemas que possam comprometer a confiabilidade da análise exploratória.

Serão investigados valores ausentes, registros duplicados, inconsistências, valores inválidos e outliers.

A limpeza será realizada de forma criteriosa, preservando as informações relevantes e documentando as decisões tomadas.

### 4.1 Diagnóstico de valores ausentes

Inicialmente, será realizada a identificação dos valores nulos (`NaN`) presentes em cada conjunto de dados.

Serão utilizadas as funções:

- `isna().sum()`: quantidade de valores ausentes por coluna.
- `isna().mean() * 100`: percentual de valores ausentes.
- `nunique()`: quantidade de valores distintos.
- `value_counts()`: frequência das categorias, incluindo valores ausentes.

O objetivo é compreender a distribuição dos dados faltantes antes de decidir como tratá-los.

### 4.2 Identificação de registros duplicados

Será verificada a existência de registros duplicados utilizando `duplicated()`.

A análise considerará duas situações:

- **Duplicação completa:** registros com todas as informações iguais.
- **Duplicação por chave:** repetição de identificadores, como `order_id`, `product_id` e `customer_id`.

Como um pedido pode conter vários produtos, a repetição de `order_id` na base de itens não será considerada automaticamente um erro.

### 4.3 Verificação de inconsistências

Serão investigadas possíveis inconsistências nos dados, incluindo:

- Diferenças de escrita em categorias.
- Espaços desnecessários em campos de texto.
- Tipos de dados incompatíveis.
- Datas inválidas ou incoerentes.
- Valores numéricos negativos ou fora dos limites esperados.
- Possíveis códigos utilizados para representar informações ausentes.

Quando necessário, serão aplicadas técnicas de padronização e conversão de tipos.

### 4.4 Análise de valores extremos (outliers)

Serão investigados valores extremos nas variáveis numéricas relevantes, especialmente nos preços dos produtos e nos valores de frete.

Serão utilizadas estatísticas descritivas, quartis e o intervalo interquartil (IQR) para identificar observações potencialmente atípicas.

Os outliers não serão removidos automaticamente, pois valores elevados podem representar transações legítimas.

### 4.5 Tratamento dos dados

Após o diagnóstico, serão aplicados apenas os tratamentos necessários, considerando:

- Manutenção de valores ausentes quando não comprometerem a análise.
- Exclusão de registros somente quando houver justificativa.
- Correção ou padronização de inconsistências identificadas.
- Conversão adequada de colunas de datas.
- Preservação dos registros originais para comparação.

As alterações serão realizadas em cópias dos DataFrames, evitando modificações irreversíveis nas bases originais.

### 4.6 Relatório de limpeza

Será elaborado um relatório contendo:

| Informação | Descrição |
|---|---|
| Problema identificado | Tipo de inconsistência encontrada |
| Quantidade afetada | Número de registros envolvidos |
| Tratamento aplicado | Procedimento realizado |
| Justificativa | Motivo da decisão |
| Impacto | Alterações provocadas na base |

### 4.7 Validação após a limpeza

Ao final, serão realizadas novas verificações para avaliar a qualidade dos dados tratados.

Serão comparados os resultados antes e depois da limpeza, incluindo:

- Quantidade de linhas e colunas.
- Quantidade de valores ausentes.
- Registros duplicados.
- Tipos das variáveis.
- Estatísticas descritivas relevantes.

### Resultado esperado

Obter conjuntos de dados consistentes e adequados para as próximas etapas da análise exploratória, mantendo a rastreabilidade das transformações realizadas.

**Princípio adotado:** diagnosticar antes de tratar, justificar cada decisão e preservar informações relevantes para responder às perguntas do projeto.


In [ ]:
#Diagnosticando valores ausentes
#Esse código apresenta, para cada coluna, seu tipo de dado, quantidade e percentual de valores ausentes e número de valores distintos.


for nome, df in bases.items():
    print(f"\n===== DIAGNÓSTICO: {nome.upper()} =====")

    diagnostico = pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "faltantes": df.isna().sum(),
        "percentual_faltantes": (
            df.isna().mean() * 100
        ).round(2),
        "valores_distintos": df.nunique()
    })

    display(diagnostico)



#### 4.1.1 Diagnóstico de valores ausentes — Base de Pedidos

A base de pedidos contém 99.441 registros e 8 variáveis.

A análise de valores ausentes foi realizada utilizando os métodos `isna().sum()` e `isna().mean() * 100`.

**Resultados encontrados:**

- `order_approved_at`: 160 valores ausentes (0,16%).
- `order_delivered_carrier_date`: 1.783 valores ausentes (1,79%).
- `order_delivered_customer_date`: 2.965 valores ausentes (2,98%).

As demais variáveis não apresentaram valores ausentes.

**Interpretação:**

Os valores ausentes estão concentrados nas datas relacionadas às etapas de processamento e entrega dos pedidos.

Essas ausências podem estar associadas a pedidos cancelados, não entregues ou que não concluíram determinadas etapas do processo.

**Decisão preliminar:**

Os registros serão preservados nesta fase, pois ainda não existem evidências suficientes para justificar sua exclusão.

Será realizada uma análise cruzada entre os valores ausentes e o status dos pedidos, permitindo identificar se as ausências são coerentes com a situação de cada pedido.


### 4.1.2 Diagnóstico de valores ausentes — Produtos e Clientes

#### Base de Produtos

A base contém 32.951 registros e 9 variáveis.

Foram identificados os seguintes valores ausentes:

- Categoria do produto: 610 registros (1,85%).
- Comprimento do nome: 610 registros (1,85%).
- Comprimento da descrição: 610 registros (1,85%).
- Quantidade de fotos: 610 registros (1,85%).
- Peso e dimensões: 2 registros ausentes por variável (0,01%).

**Interpretação:**

A maior concentração de valores ausentes está nas informações descritivas dos produtos.

A ausência da categoria merece atenção, pois uma das perguntas do projeto busca identificar quais categorias apresentam maior valor de vendas.

**Decisão de tratamento:**

Os produtos sem categoria serão mantidos e identificados como `categoria_nao_informada`, permitindo sua inclusão nas análises sem atribuir categorias não comprovadas.

Os valores ausentes em peso, dimensões e demais atributos descritivos serão preservados, pois não são necessários para responder às perguntas principais.

#### Base de Clientes

A base contém 99.441 registros e 5 variáveis.

Não foram identificados valores ausentes.

Foram encontrados 99.441 identificadores `customer_id` distintos e 96.096 identificadores `customer_unique_id` distintos.

**Interpretação:**

A repetição de `customer_unique_id` pode representar consumidores que realizaram mais de uma compra, não caracterizando necessariamente duplicidade indevida.

**Decisão de tratamento:**

Não será realizado tratamento de valores ausentes na base de clientes.

Os registros serão preservados, mantendo a possibilidade de analisar pedidos associados a um mesmo consumidor.


In [ ]:

# Verificando registros totalmente duplicados nas quatro bases

for nome, df in bases.items():
    print(f"\n===== {nome.upper()} =====")

    duplicados = df.duplicated().sum()

    print(f"Registros totalmente duplicados: {duplicados}")



#### Resultado da verificação de duplicados completos

Foi realizada a verificação de registros totalmente duplicados nas quatro bases da Olist, utilizando o método `duplicated().sum()` da biblioteca Pandas.

**Resultados obtidos:**

| Base de dados | Registros duplicados |
|---|---:|
| Pedidos | 0 |
| Itens | 0 |
| Produtos | 0 |
| Clientes | 0 |

**Interpretação:**

Nenhuma das quatro bases apresentou registros totalmente duplicados.

**Decisão de limpeza:**

Não foi necessária a remoção de registros por duplicação completa, preservando integralmente os dados originais.

A próxima verificação será realizada sobre as chaves de identificação, pois a ausência de linhas totalmente duplicadas não garante que os identificadores sejam únicos.


In [ ]:

# Verificando duplicações nas chaves de identificação

chaves = {
    "Pedidos": (df_orders, ["order_id"]),
    "Itens": (df_items, ["order_id", "order_item_id"]),
    "Produtos": (df_products, ["product_id"]),
    "Clientes": (df_customers, ["customer_id"])
}

for nome, (df, colunas) in chaves.items():

    duplicados = df.duplicated(
        subset=colunas
    ).sum()

    print(f"{nome}: {duplicados} chaves duplicadas")


In [ ]:

# Relatório consolidado de valores ausentes

relatorios = []

for nome, df in bases.items():

    tabela = pd.DataFrame({
        "base": nome,
        "coluna": df.columns,
        "faltantes": df.isna().sum().values,
        "percentual": (
            df.isna().mean() * 100
        ).round(2).values
    })

    relatorios.append(tabela)

relatorio_faltantes = pd.concat(
    relatorios,
    ignore_index=True
)

display(
    relatorio_faltantes[
        relatorio_faltantes["faltantes"] > 0
    ].sort_values(
        "faltantes",
        ascending=False
    )
)



### 4.1.3 Relatório consolidado de valores ausentes

Foi elaborado um relatório consolidado utilizando as quatro bases da Olist, com o objetivo de identificar as variáveis que apresentam dados ausentes.

**Principais resultados:**

- A base de pedidos apresenta valores ausentes nas datas de aprovação, envio à transportadora e entrega ao cliente.
- A base de produtos apresenta 610 registros sem categoria e sem algumas informações descritivas.
- Apenas 2 registros apresentam ausência em cada variável de peso e dimensões dos produtos.
- As bases de itens e clientes não apresentam valores ausentes.

**Interpretação:**

Os valores ausentes estão concentrados principalmente nas informações logísticas dos pedidos e nos atributos descritivos dos produtos.

**Decisão:**

Os registros serão preservados inicialmente. Os tratamentos serão aplicados apenas após a investigação de cada situação, considerando a relevância das variáveis para as perguntas do projeto.

Esse procedimento permite evitar exclusões desnecessárias e manter a rastreabilidade das decisões de limpeza.


In [ ]:

# Investigando datas ausentes por status do pedido

colunas_datas = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
]

for coluna in colunas_datas:

    print(f"\n===== {coluna} =====")

    tabela = pd.crosstab(
        df_orders["order_status"],
        df_orders[coluna].isna(),
        margins=True
    )

    display(tabela)



### 4.1.4 Investigação de valores ausentes por status dos pedidos

Para compreender a origem dos valores ausentes nas datas dos pedidos, foi realizada uma análise cruzada entre `order_status` e as variáveis de aprovação, envio à transportadora e entrega ao cliente, utilizando `pd.crosstab()`.

#### Resultados encontrados

**Data de aprovação (`order_approved_at`):**

Foram identificados 160 registros sem data de aprovação, distribuídos entre:

- 141 pedidos cancelados.
- 5 pedidos com status `created`.
- 14 pedidos com status `delivered`.

**Data de envio (`order_delivered_carrier_date`):**

Foram identificados 1.783 registros sem data de envio.

A maioria corresponde a pedidos que não chegaram à etapa de transporte ou que foram cancelados. Entretanto, 2 pedidos com status `delivered` não possuem data de envio.

**Data de entrega (`order_delivered_customer_date`):**

Foram identificados 2.965 registros sem data de entrega.

A maior parte corresponde a pedidos sem status de entrega concluída. Entretanto, 8 pedidos classificados como `delivered` apresentam ausência dessa informação.

#### Interpretação

Os resultados indicam que grande parte dos valores ausentes está relacionada às diferentes etapas do processamento dos pedidos.

Entretanto, foram identificadas inconsistências pontuais em pedidos classificados como entregues, mas sem determinadas datas de processamento ou entrega.

Essas inconsistências não serão corrigidas artificialmente, pois não existem informações suficientes para determinar as datas reais.

#### Decisão de limpeza

- Preservar os valores ausentes nas datas de aprovação, envio e entrega.
- Não substituir datas ausentes por valores estimados.
- Manter os registros originais para garantir a rastreabilidade.
- Utilizar a data da compra como referência temporal para a análise de vendas.
- Filtrar os pedidos com status `delivered` na etapa de preparação da base analítica.

**Justificativa:**

A manutenção dos registros evita perdas desnecessárias de informações comerciais. As datas ausentes não impedem diretamente as análises de valores de vendas, estados e categorias propostas neste projeto.

Para eventuais análises de prazo de entrega, será necessário considerar apenas registros com datas válidas.

**Impacto do tratamento:**

Nenhum registro foi excluído ou teve suas datas modificadas nesta etapa.



### 4.8 Investigação de duplicados e valores ausentes

Foram realizadas verificações de duplicidade completa e por chaves de identificação nas quatro bases.

Na base de itens, a repetição de `order_id` é esperada, pois um pedido pode conter múltiplos produtos. Por isso, a identificação de cada item considera a combinação de `order_id` e `order_item_id`.

Também foram analisados os valores ausentes por coluna, incluindo a relação entre as datas não preenchidas e os diferentes status dos pedidos.

Nesta fase, nenhum registro foi excluído automaticamente.

As decisões de tratamento serão tomadas após a avaliação dos resultados, preservando os dados necessários para responder às perguntas do projeto.


In [ ]:

# Pedidos entregues com datas incompletas

colunas_verificar = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
]

pedidos_entregues = df_orders.loc[
    df_orders["order_status"] == "delivered"
].copy()

pedidos_incompletos = pedidos_entregues.loc[
    pedidos_entregues[colunas_verificar].isna().any(axis=1)
]

print("Total de pedidos entregues:", len(pedidos_entregues))

print(
    "Pedidos entregues com alguma data ausente:",
    len(pedidos_incompletos)
)

display(
    pedidos_incompletos[
        ["order_id", "order_status"] + colunas_verificar
    ]
)



### 4.1.5 Validação de datas incompletas em pedidos entregues

Foi realizada uma investigação específica dos pedidos classificados com status `delivered`, verificando a presença de valores ausentes nas datas de aprovação, envio à transportadora e entrega ao cliente.

#### Resultados

- Total de pedidos entregues: 96.478.
- Pedidos entregues com pelo menos uma data ausente: 23.
- Percentual de pedidos afetados: aproximadamente 0,024%.

Foram identificadas 24 ocorrências de datas ausentes, distribuídas em 23 pedidos distintos, pois um pedido apresenta ausência em mais de uma variável.

#### Interpretação

Embora os pedidos estejam classificados como entregues, alguns não possuem todas as datas correspondentes às etapas de processamento e entrega.

Essas situações indicam possíveis falhas ou limitações no registro das informações logísticas.

#### Decisão de limpeza

Os 23 pedidos serão preservados, pois possuem status de entrega concluída e podem contribuir para a análise comercial.

As datas ausentes não serão preenchidas artificialmente.

#### Justificativa

A análise principal utiliza a data da compra, os valores dos pedidos, os estados dos clientes e as categorias dos produtos.

Portanto, a ausência de algumas datas logísticas não impede a realização das análises propostas.

Caso sejam realizadas análises de prazo de entrega, os registros sem as datas necessárias deverão ser tratados separadamente.

#### Impacto

- Registros removidos: 0.
- Datas preenchidas artificialmente: 0.
- Pedidos preservados: 96.478.

A decisão permite manter a integridade da base e documentar as limitações identificadas.


In [ ]:

# Diagnóstico de valores inválidos

print("===== ITENS =====")

print(
    "Preços negativos:",
    (df_items["price"] < 0).sum()
)

print(
    "Fretes negativos:",
    (df_items["freight_value"] < 0).sum()
)

print(
    "Preços iguais a zero:",
    (df_items["price"] == 0).sum()
)

print(
    "Fretes iguais a zero:",
    (df_items["freight_value"] == 0).sum()
)

print("\n===== PRODUTOS =====")

colunas_numericas = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for coluna in colunas_numericas:

    negativos = (df_products[coluna] < 0).sum()
    zeros = (df_products[coluna] == 0).sum()

    print(f"\n{coluna}")
    print("Valores negativos:", negativos)
    print("Valores iguais a zero:", zeros)



### 4.3.1 Diagnóstico de valores numéricos inválidos

Foi realizada uma verificação de valores negativos e iguais a zero nas variáveis numéricas das bases de itens e produtos.

#### Resultados encontrados

**Base de Itens:**

- Preços negativos: 0.
- Preços iguais a zero: 0.
- Fretes negativos: 0.
- Fretes iguais a zero: 383.

**Base de Produtos:**

- Pesos negativos: 0.
- Pesos iguais a zero: 4.
- Dimensões negativas: 0.
- Dimensões iguais a zero: 0.

#### Interpretação

Não foram encontrados preços negativos ou iguais a zero, nem valores negativos de frete, peso ou dimensões.

Foram identificados 383 itens com frete igual a zero, situação que pode representar frete gratuito ou outra condição comercial.

Também foram identificados 4 produtos com peso igual a zero, o que merece investigação por representar uma possível inconsistência cadastral.

#### Decisão preliminar

Os registros serão preservados até a conclusão da investigação.

Não serão realizadas substituições automáticas de valores iguais a zero, evitando a criação de informações sem comprovação.

#### Impacto

Até o momento, nenhum registro foi excluído ou alterado em decorrência dessas verificações.


In [ ]:

#Investigar produtos com peso igual a zero

produtos_peso_zero = df_products.loc[
    df_products["product_weight_g"] == 0
].copy()

print(
    "Quantidade de produtos com peso zero:",
    len(produtos_peso_zero)
)

display(
    produtos_peso_zero[
        [
            "product_id",
            "product_category_name",
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    ]
)



### 4.3.2 Investigação de produtos com peso igual a zero

Foram identificados 4 produtos com peso cadastrado igual a zero.

Todos pertencem à categoria `cama_mesa_banho` e apresentam as mesmas dimensões:

- Comprimento: 30 cm.
- Altura: 25 cm.
- Largura: 30 cm.
- Peso: 0 g.

#### Interpretação

A existência de dimensões preenchidas e peso igual a zero sugere uma possível inconsistência no cadastro dos produtos.

Entretanto, não existem informações suficientes para determinar o peso real desses itens.

#### Decisão de limpeza

Os quatro registros serão preservados, mantendo o peso original igual a zero.

Não será realizada imputação de valores, pois isso exigiria estimativas sem comprovação.

#### Justificativa

As variáveis de peso e dimensões não são necessárias para responder às perguntas principais deste projeto, que estão relacionadas às vendas, aos valores dos pedidos, aos estados e às categorias de produtos.

#### Impacto

- Produtos investigados: 4.
- Registros excluídos: 0.
- Valores modificados: 0.

A inconsistência será documentada como uma limitação de qualidade dos dados.


In [ ]:

#Diagnóstico de outliers pelo método IQR

colunas_analisar = ["price", "freight_value"]

for coluna in colunas_analisar:

    print(f"\n===== OUTLIERS: {coluna.upper()} =====")

    q1 = df_items[coluna].quantile(0.25)
    q3 = df_items[coluna].quantile(0.75)

    iqr = q3 - q1

    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    outliers = df_items.loc[
        (df_items[coluna] < limite_inferior) |
        (df_items[coluna] > limite_superior)
    ]

    print(f"Q1: R$ {q1:.2f}")
    print(f"Q3: R$ {q3:.2f}")
    print(f"IQR: R$ {iqr:.2f}")

    print(f"Limite inferior: R$ {limite_inferior:.2f}")
    print(f"Limite superior: R$ {limite_superior:.2f}")

    print(f"Quantidade de possíveis outliers: {len(outliers)}")

    percentual = len(outliers) / len(df_items) * 100

    print(f"Percentual: {percentual:.2f}%")



### 4.4.1 Identificação de Outliers pelo Método IQR

Foi realizada uma investigação de valores extremos nas variáveis `price` e `freight_value`, utilizando o método do Intervalo Interquartil (IQR).

O método considera os quartis Q1 e Q3 e estabelece limites de 1,5 vez o intervalo interquartil para identificar observações potencialmente atípicas.

#### Resultados — Preços

- Primeiro quartil (Q1): R$ 39,90.
- Terceiro quartil (Q3): R$ 134,90.
- Intervalo interquartil (IQR): R$ 95,00.
- Limite superior: R$ 277,40.
- Possíveis outliers: 8.427 registros (7,48%).

#### Resultados — Fretes

- Primeiro quartil (Q1): R$ 13,08.
- Terceiro quartil (Q3): R$ 21,15.
- Intervalo interquartil (IQR): R$ 8,07.
- Limite inferior: R$ 0,98.
- Limite superior: R$ 33,25.
- Possíveis outliers: 12.134 registros (10,77%).

#### Interpretação

A distribuição dos preços apresenta uma parcela de produtos com valores significativamente superiores à faixa central dos dados.

Os fretes também apresentam valores extremos, incluindo fretes gratuitos e valores elevados.

Essas observações podem representar características reais das transações comerciais, não sendo necessariamente erros cadastrais.

#### Decisão preliminar

Os possíveis outliers serão preservados até a conclusão da investigação dos valores extremos e de seu impacto nas estatísticas descritivas.

A identificação estatística de um outlier não constitui justificativa suficiente para sua exclusão.


In [ ]:

#Investigação dos valores extremos

print("===== ESTATÍSTICAS DOS PREÇOS E FRETES =====")

display(
    df_items[["price", "freight_value"]].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    ).T
)

print("\n===== 10 MAIORES PREÇOS =====")

display(
    df_items.nlargest(10, "price")[
        ["order_id", "product_id", "price", "freight_value"]
    ]
)

print("\n===== 10 MAIORES FRETES =====")

display(
    df_items.nlargest(10, "freight_value")[
        ["order_id", "product_id", "price", "freight_value"]
    ]
)



### 4.4.2 Investigação dos valores extremos e decisão de tratamento

Após a identificação dos possíveis outliers pelo método IQR, foram analisadas as estatísticas descritivas e as dez maiores observações das variáveis `price` e `freight_value`.

#### Análise dos preços

- Média: R$ 120,65.
- Mediana: R$ 74,99.
- Desvio padrão: R$ 183,63.
- Percentil 95: R$ 349,90.
- Percentil 99: R$ 890,00.
- Valor máximo: R$ 6.735,00.

A média superior à mediana indica assimetria positiva na distribuição dos preços.

Os valores extremos podem representar produtos de maior valor comercial, não sendo necessariamente erros de cadastro.

#### Análise dos fretes

- Média: R$ 19,99.
- Mediana: R$ 16,26.
- Desvio padrão: R$ 15,81.
- Percentil 95: R$ 45,12.
- Percentil 99: R$ 84,52.
- Valor máximo: R$ 409,68.

A distribuição dos fretes também apresenta assimetria à direita, com valores elevados em uma parcela das transações.

#### Decisão de limpeza

Os possíveis outliers identificados serão mantidos nas bases de dados.

#### Justificativa

A identificação estatística de valores extremos não comprova a existência de erros.

Como o objetivo do projeto é investigar o comportamento das vendas, a exclusão desses registros poderia subestimar os valores comercializados e alterar a distribuição dos preços.

Nas análises posteriores, serão utilizadas medidas como média, mediana, quartis e percentis para interpretar adequadamente as distribuições.

#### Impacto da decisão

- Outliers de preço identificados: 8.427.
- Outliers de frete identificados: 12.134.
- Registros excluídos: 0.
- Valores modificados: 0.

Os valores extremos foram documentados e preservados.


In [ ]:

# Relatório de limpeza dos dados

relatorio_limpeza = pd.DataFrame([
    {
        "base": "Pedidos",
        "problema": "Datas ausentes",
        "registros_afetados": 23,
        "tratamento": "Manutenção dos valores ausentes",
        "justificativa": "Pedidos entregues preservados; datas não inventadas"
    },
    {
        "base": "Produtos",
        "problema": "Categorias ausentes",
        "registros_afetados": 610,
        "tratamento": "Identificação como categoria_nao_informada",
        "justificativa": "Preservar produtos sem atribuir categoria desconhecida"
    },
    {
        "base": "Produtos",
        "problema": "Peso igual a zero",
        "registros_afetados": 4,
        "tratamento": "Manutenção dos valores originais",
        "justificativa": "Peso não utilizado nas análises de vendas"
    },
    {
        "base": "Itens",
        "problema": "Frete igual a zero",
        "registros_afetados": 383,
        "tratamento": "Manutenção dos valores originais",
        "justificativa": "Possibilidade de frete gratuito"
    },
    {
        "base": "Itens",
        "problema": "Outliers de preço",
        "registros_afetados": 8427,
        "tratamento": "Manutenção dos valores extremos",
        "justificativa": "Podem representar vendas legítimas"
    },
    {
        "base": "Itens",
        "problema": "Outliers de frete",
        "registros_afetados": 12134,
        "tratamento": "Manutenção dos valores extremos",
        "justificativa": "Não há evidência suficiente de erro"
    }
])

display(relatorio_limpeza)



### 4.5 Relatório de Limpeza dos Dados

Após a realização dos diagnósticos de qualidade, foi elaborado um relatório consolidado contendo os problemas identificados, a quantidade de registros afetados, os tratamentos definidos e suas respectivas justificativas.

O objetivo é documentar as decisões de limpeza, garantindo transparência, rastreabilidade e reprodutibilidade da análise.

#### 4.5.1 Resumo dos problemas identificados

| Base | Problema identificado | Registros afetados | Tratamento |
|---|---|---:|---|
| Pedidos | Datas ausentes em pedidos entregues | 23 | Preservação dos valores ausentes |
| Produtos | Categorias ausentes | 610 | Identificação como categoria não informada |
| Produtos | Peso igual a zero | 4 | Manutenção dos valores originais |
| Itens | Frete igual a zero | 383 | Manutenção dos valores originais |
| Itens | Outliers de preço | 8.427 | Manutenção dos valores extremos |
| Itens | Outliers de frete | 12.134 | Manutenção dos valores extremos |

#### 4.5.2 Justificativas das decisões

**Datas ausentes nos pedidos:**

Foram identificados 23 pedidos entregues com pelo menos uma data logística ausente. Os registros foram preservados, pois a análise comercial utiliza a data da compra, que está preenchida.

**Categorias ausentes nos produtos:**

Foram identificados 610 produtos sem categoria. Foi definida a substituição dos valores ausentes por `categoria_nao_informada`, sem atribuir categorias desconhecidas.

**Produtos com peso igual a zero:**

Os quatro produtos foram preservados, pois o peso não é necessário para responder às perguntas principais do projeto.

**Fretes iguais a zero:**

Os 383 registros foram mantidos, pois podem representar condições comerciais de frete gratuito.

**Outliers de preço e frete:**

Foram identificados 8.427 possíveis outliers de preço e 12.134 de frete pelo método IQR.

Esses registros foram preservados, pois valores estatisticamente extremos não representam necessariamente erros.

#### 4.5.3 Impacto das decisões de limpeza

As decisões priorizaram a preservação dos registros e evitaram exclusões sem justificativa.

Até esta etapa:

- Nenhum registro foi excluído.
- Nenhuma data ausente foi preenchida artificialmente.
- Os valores extremos foram preservados.
- Os valores iguais a zero foram mantidos.
- A identificação das categorias ausentes foi definida como tratamento necessário para a análise por categoria.

A validação da aplicação desse tratamento será apresentada na próxima seção.

#### 4.5.4 Considerações finais

A investigação demonstrou que nem todos os valores ausentes, extremos ou iguais a zero representam erros.

A estratégia de limpeza adotada buscou preservar a integridade das informações comerciais e documentar as limitações identificadas.

As decisões serão verificadas por meio da comparação entre os dados originais e os dados preparados para análise.


In [ ]:

# Validação da limpeza dos produtos

# Recriar a cópia limpa a partir da base original
df_products_clean = df_products.copy()

# Padronizar categorias existentes
df_products_clean["product_category_name"] = (
    df_products_clean["product_category_name"]
    .astype("string")
    .str.strip()
    .str.lower()
    .fillna("categoria_nao_informada")
)

# Comparar antes e depois
comparacao = pd.DataFrame({
    "indicador": [
        "Quantidade de registros",
        "Categorias ausentes",
        "Categorias não informadas",
        "Produtos com peso zero"
    ],
    "antes": [
        len(df_products),
        df_products["product_category_name"].isna().sum(),
        0,
        (df_products["product_weight_g"] == 0).sum()
    ],
    "depois": [
        len(df_products_clean),
        df_products_clean["product_category_name"].isna().sum(),
        (
            df_products_clean["product_category_name"]
            == "categoria_nao_informada"
        ).sum(),
        (df_products_clean["product_weight_g"] == 0).sum()
    ]
})

display(comparacao)



### 4.6 Validação dos tratamentos realizados

Após a investigação dos problemas de qualidade, foi realizada a validação da base de produtos.

A limpeza foi aplicada sobre uma cópia do conjunto original, preservando os dados brutos.

#### Tratamentos realizados

- Padronização dos nomes das categorias com `strip()` e `lower()`.
- Substituição de 610 categorias ausentes por `categoria_nao_informada`.
- Preservação dos quatro produtos com peso igual a zero.
- Manutenção dos valores extremos de preço e frete.
- Preservação das datas logísticas ausentes.

#### Justificativa

A estratégia adotada priorizou a conservação dos registros comerciais, evitando exclusões e imputações sem evidências suficientes.

Os valores ausentes foram tratados apenas quando necessário para permitir as análises propostas.

#### Validação

A quantidade de produtos foi mantida em 32.951 registros.

As 610 categorias ausentes passaram a ser identificadas explicitamente como `categoria_nao_informada`, sem atribuição de categorias fictícias.

A base original permaneceu preservada para comparação e rastreabilidade.



### 4.6.1 Validação da limpeza da base de produtos

Após a aplicação dos tratamentos, foi realizada uma comparação entre a base original e a base preparada.

#### Resultados da validação

| Indicador | Antes | Depois |
|---|---:|---:|
| Quantidade de registros | 32.951 | 32.951 |
| Categorias ausentes | 610 | 0 |
| Categorias não informadas | 0 | 610 |
| Produtos com peso zero | 4 | 4 |

#### Interpretação

A comparação demonstra que a limpeza das categorias foi realizada sem exclusão de produtos.

Os 610 registros originalmente sem categoria passaram a ser identificados como `categoria_nao_informada`.

Os quatro produtos com peso igual a zero foram preservados, conforme a decisão de tratamento.

#### Conclusão

O tratamento foi aplicado com sucesso, preservando a quantidade original de produtos e permitindo identificar explicitamente as categorias desconhecidas.

A base `df_products_clean` será utilizada nas próximas etapas da análise exploratória.


In [ ]:

# Verificar integridade das chaves

verificacoes = {
    "Pedidos sem cliente correspondente":
        ~df_orders["customer_id"].isin(df_customers["customer_id"]),

    "Itens sem pedido correspondente":
        ~df_items["order_id"].isin(df_orders["order_id"]),

    "Itens sem produto correspondente":
        ~df_items["product_id"].isin(df_products_clean["product_id"])
}

bases_verificacao = {
    "Pedidos sem cliente correspondente": df_orders,
    "Itens sem pedido correspondente": df_items,
    "Itens sem produto correspondente": df_items
}

for nome, condicao in verificacoes.items():
    quantidade = condicao.sum()
    print(f"{nome}: {quantidade}")



### 4.7 Verificação da Integridade Referencial

Antes de realizar a integração das bases de dados, foi verificada a integridade das chaves utilizadas nos relacionamentos.

A análise foi realizada utilizando o método `isin()` da biblioteca Pandas, permitindo identificar registros sem correspondência entre as tabelas.

#### Resultados encontrados

| Verificação | Registros sem correspondência |
|---|---:|
| Pedidos sem cliente correspondente | 0 |
| Itens sem pedido correspondente | 0 |
| Itens sem produto correspondente | 0 |

#### Interpretação

Não foram identificados registros sem correspondência entre as chaves analisadas.

Todos os pedidos possuem clientes cadastrados, todos os itens possuem pedidos correspondentes e todos os produtos associados aos itens estão presentes na base de produtos.

#### Decisão de limpeza

Não foi necessária nenhuma exclusão ou correção de identificadores.

#### Justificativa

A integridade das chaves permite relacionar as bases utilizando operações de `merge()`, sem perda de registros decorrente de identificadores inexistentes.

#### Impacto

- Registros inconsistentes identificados: 0.
- Registros excluídos: 0.
- Identificadores modificados: 0.

A verificação contribui para a confiabilidade da integração dos dados na próxima etapa.


In [ ]:

# Verificar consistência cronológica das datas

# Criar uma cópia para preservar os dados originais
df_orders_clean = df_orders.copy()

# Converter colunas de datas para datetime
colunas_datas = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for coluna in colunas_datas:
    df_orders_clean[coluna] = pd.to_datetime(
        df_orders_clean[coluna],
        errors="coerce"
    )

# Verificar datas inválidas geradas na conversão
print("===== DATAS AUSENTES APÓS CONVERSÃO =====")

for coluna in colunas_datas:
    print(
        f"{coluna}:",
        df_orders_clean[coluna].isna().sum()
    )

# Verificar inconsistências cronológicas
print("\n===== INCONSISTÊNCIAS CRONOLÓGICAS =====")

aprovacao_antes_compra = (
    df_orders_clean["order_approved_at"]
    < df_orders_clean["order_purchase_timestamp"]
).sum()

envio_antes_compra = (
    df_orders_clean["order_delivered_carrier_date"]
    < df_orders_clean["order_purchase_timestamp"]
).sum()

entrega_antes_envio = (
    df_orders_clean["order_delivered_customer_date"]
    < df_orders_clean["order_delivered_carrier_date"]
).sum()

print("Aprovação antes da compra:", aprovacao_antes_compra)
print("Envio antes da compra:", envio_antes_compra)
print("Entrega antes do envio:", entrega_antes_envio)



### 4.8 Verificação da Consistência Cronológica

Foi realizada a conversão das variáveis de datas para o tipo `datetime`, utilizando `pd.to_datetime()` com o parâmetro `errors="coerce"`.

A transformação foi aplicada sobre uma cópia da base original, denominada `df_orders_clean`.

#### Validação da conversão

Após a conversão, os valores ausentes permaneceram inalterados:

- Data da compra: 0.
- Data de aprovação: 160.
- Data de envio à transportadora: 1.783.
- Data de entrega ao cliente: 2.965.
- Data estimada de entrega: 0.

Isso indica que não foram introduzidos novos valores ausentes durante a conversão.

#### Inconsistências cronológicas identificadas

Foram realizadas três verificações:

| Verificação | Registros identificados |
|---|---:|
| Aprovação anterior à compra | 0 |
| Envio anterior à compra | 166 |
| Entrega anterior ao envio | 23 |

#### Interpretação

Não foram identificadas aprovações anteriores à data da compra.

Entretanto, foram encontrados registros em que a data de envio é anterior à compra e registros em que a entrega ocorreu antes da data de envio.

Essas situações representam possíveis inconsistências cronológicas, que podem estar relacionadas a problemas de registro ou diferenças entre sistemas.

#### Decisão de limpeza

Os registros serão preservados nesta etapa.

As datas não serão corrigidas artificialmente, pois não existem informações suficientes para determinar os horários reais dos eventos.

#### Justificativa

A análise principal utilizará a data da compra, que não apresenta valores ausentes, e terá como foco os valores das vendas, os estados dos clientes e as categorias dos produtos.

As inconsistências identificadas deverão ser consideradas como limitações caso sejam realizadas análises de desempenho logístico.

#### Impacto

- Registros excluídos: 0.
- Datas modificadas artificialmente: 0.
- Colunas convertidas para `datetime`: 5.

As inconsistências foram documentadas para garantir transparência e rastreabilidade.


In [ ]:

# Investigar diferenças cronológicas

# Pedidos enviados antes da compra
envio_antes_compra = df_orders_clean.loc[
    df_orders_clean["order_delivered_carrier_date"]
    < df_orders_clean["order_purchase_timestamp"]
].copy()

# Calcular diferença em horas
envio_antes_compra["diferenca_horas"] = (
    (
        envio_antes_compra["order_purchase_timestamp"]
        - envio_antes_compra["order_delivered_carrier_date"]
    ).dt.total_seconds() / 3600
)

print("===== ENVIO ANTES DA COMPRA =====")

display(
    envio_antes_compra["diferenca_horas"].describe()
)

display(
    envio_antes_compra[
        [
            "order_id",
            "order_purchase_timestamp",
            "order_delivered_carrier_date",
            "diferenca_horas"
        ]
    ].sort_values("diferenca_horas", ascending=False).head(10)
)

# Pedidos entregues antes do envio
entrega_antes_envio = df_orders_clean.loc[
    df_orders_clean["order_delivered_customer_date"]
    < df_orders_clean["order_delivered_carrier_date"]
].copy()

# Calcular diferença em horas
entrega_antes_envio["diferenca_horas"] = (
    (
        entrega_antes_envio["order_delivered_carrier_date"]
        - entrega_antes_envio["order_delivered_customer_date"]
    ).dt.total_seconds() / 3600
)

print("\n===== ENTREGA ANTES DO ENVIO =====")

display(
    entrega_antes_envio["diferenca_horas"].describe()
)

display(
    entrega_antes_envio[
        [
            "order_id",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "diferenca_horas"
        ]
    ].sort_values("diferenca_horas", ascending=False).head(10)
)



### 4.8.1 Investigação das Diferenças Cronológicas

Após a identificação das inconsistências nas datas dos pedidos, foi realizada uma análise das diferenças em horas entre os eventos registrados.

O objetivo foi avaliar a magnitude das inconsistências antes de definir qualquer tratamento.

#### Envio anterior à compra

Foram identificados 166 pedidos com data de envio anterior à data da compra.

**Estatísticas:**

- Média: 26,04 horas.
- Mediana: 0,60 hora.
- Primeiro quartil: 0,23 hora.
- Terceiro quartil: 1,02 hora.
- Diferença máxima: 4.109,10 horas.

A mediana indica que metade das inconsistências apresenta diferenças de até aproximadamente 36 minutos.

Entretanto, foi identificado um caso extremo com diferença de aproximadamente 171 dias.

A diferença entre média e mediana demonstra a influência de valores extremos sobre a média.

#### Entrega anterior ao envio

Foram identificados 23 pedidos com data de entrega anterior à data de envio.

**Estatísticas:**

- Média: 78,46 horas.
- Mediana: 39,86 horas.
- Primeiro quartil: 23,34 horas.
- Terceiro quartil: 128,76 horas.
- Diferença máxima: 386,31 horas.

As diferenças são mais expressivas nesse grupo, indicando possíveis problemas de registro dos eventos logísticos.

#### Interpretação

As inconsistências podem estar relacionadas a falhas de registro, diferenças entre sistemas ou outros problemas operacionais.

Não é possível determinar a causa exata apenas com as informações disponíveis.

#### Decisão de limpeza

Os registros serão preservados, sem alterações artificiais nas datas.

#### Justificativa

O projeto tem como objetivo analisar o comportamento das vendas, a distribuição geográfica dos clientes e as categorias de produtos.

As inconsistências cronológicas não impedem diretamente essas análises, pois a data da compra está disponível em todos os pedidos.

Entretanto, esses registros exigiriam tratamento específico em uma análise de desempenho logístico.

#### Impacto

- Registros excluídos: 0.
- Datas corrigidas artificialmente: 0.
- Inconsistências documentadas: 189 ocorrências cronológicas.

As ocorrências representam 166 casos de envio anterior à compra e 23 casos de entrega anterior ao envio, não necessariamente 189 pedidos distintos.


In [ ]:

# Validação final da limpeza

# Criar cópias das bases que não precisaram de alterações
df_items_clean = df_items.copy()
df_customers_clean = df_customers.copy()

bases_originais = {
    "Pedidos": df_orders,
    "Itens": df_items,
    "Produtos": df_products,
    "Clientes": df_customers
}

bases_limpas = {
    "Pedidos": df_orders_clean,
    "Itens": df_items_clean,
    "Produtos": df_products_clean,
    "Clientes": df_customers_clean
}

resumo_validacao = []

for nome in bases_originais:

    original = bases_originais[nome]
    limpa = bases_limpas[nome]

    resumo_validacao.append({
        "base": nome,
        "linhas_antes": len(original),
        "linhas_depois": len(limpa),
        "linhas_removidas": len(original) - len(limpa),
        "faltantes_antes": original.isna().sum().sum(),
        "faltantes_depois": limpa.isna().sum().sum()
    })

resumo_validacao = pd.DataFrame(resumo_validacao)

display(resumo_validacao)



### 4.9 Validação Final e Conclusão da Limpeza

Após os procedimentos de diagnóstico, investigação e tratamento, foi realizada uma comparação entre as quatro bases originais e suas respectivas versões preparadas.

#### Resultados da validação

| Base | Registros removidos | Faltantes antes | Faltantes depois |
|---|---:|---:|---:|
| Pedidos | 0 | 4.908 | 4.908 |
| Itens | 0 | 0 | 0 |
| Produtos | 0 | 2.448 | 1.838 |
| Clientes | 0 | 0 | 0 |

#### Principais resultados

- Nenhum registro foi removido das quatro bases.
- Foram identificadas explicitamente 610 categorias originalmente ausentes.
- As datas logísticas ausentes foram preservadas.
- Os valores extremos de preço e frete foram mantidos.
- Os produtos com peso igual a zero foram preservados.
- As inconsistências cronológicas foram investigadas e documentadas.
- Não foram identificadas chaves sem correspondência entre as bases.

#### Interpretação

A redução de 610 valores ausentes na base de produtos ocorreu pela substituição dos valores nulos da categoria por `categoria_nao_informada`.

As demais ausências foram mantidas por não comprometerem diretamente as perguntas principais do projeto.

#### Conclusão da Etapa 4

Os procedimentos de limpeza permitiram preparar os dados para a análise exploratória, preservando os registros comerciais e documentando as limitações encontradas.

As bases originais permaneceram intactas, enquanto as versões preparadas serão utilizadas nas próximas etapas.

A estratégia adotada priorizou a rastreabilidade, a transparência e a justificativa das decisões, evitando exclusões e correções sem evidências suficientes.



## 5. Transformação e Preparação das Variáveis

### Objetivo

Esta etapa tem como objetivo transformar e integrar os dados preparados da Olist, criando variáveis que permitam investigar o comportamento das vendas no comércio eletrônico brasileiro.

### Procedimentos planejados

1. Selecionar os pedidos com status `delivered`.
2. Relacionar as bases de pedidos, itens, produtos e clientes.
3. Criar a variável de valor de venda a partir dos preços dos itens, sem incluir o frete.
4. Calcular o valor total de produtos por pedido.
5. Extrair ano e mês das datas de compra.
6. Preparar as informações de estado e categoria para as comparações.
7. Validar a quantidade de registros e os valores calculados após as transformações.

### Unidade de análise

A integração inicial será realizada no nível dos itens dos pedidos.

Posteriormente, os valores serão agregados por pedido, mês, estado ou categoria, conforme a pergunta investigada.

Essa distinção é importante para evitar a contagem duplicada de pedidos que possuem mais de um item.

### Resultado esperado

Obter uma base analítica consistente, contendo as informações necessárias para realizar análises temporais, geográficas e comerciais.


In [ ]:

# Selecionar pedidos entregues

df_orders_delivered = df_orders_clean.loc[
    df_orders_clean["order_status"] == "delivered"
].copy()

print("Total de pedidos na base original:", len(df_orders_clean))

print("Total de pedidos entregues:", len(df_orders_delivered))

print(
    "Pedidos com outros status:",
    len(df_orders_clean) - len(df_orders_delivered)
)

print(
    "Percentual de pedidos entregues:",
    round(
        len(df_orders_delivered) / len(df_orders_clean) * 100,
        2
    ),
    "%"
)



### 5.1 Seleção dos Pedidos Entregues

Para a construção da base analítica, foram selecionados os pedidos com status `delivered`, utilizando o método `loc()` da biblioteca Pandas.

#### Resultados

- Total de pedidos: 99.441.
- Pedidos entregues: 96.478.
- Pedidos com outros status: 2.963.
- Percentual de pedidos entregues: 97,02%.

#### Interpretação

A maior parte dos pedidos registrados na base possui status de entrega concluída.

#### Decisão de preparação

Foram selecionados exclusivamente os pedidos entregues para a análise principal de vendas.

Essa seleção não representa uma exclusão por problemas de qualidade, mas um recorte analítico coerente com o objetivo do projeto.

Os pedidos com outros status permanecem disponíveis na base original.

#### Resultado

Foi criado o DataFrame `df_orders_delivered`, contendo 96.478 pedidos entregues, que será utilizado na integração com as demais bases.


In [ ]:

# Integrar pedidos entregues com itens

df_vendas = pd.merge(
    df_orders_delivered,
    df_items_clean,
    on="order_id",
    how="inner",
    validate="one_to_many"
)

print("===== INTEGRAÇÃO PEDIDOS + ITENS =====")

print("Pedidos entregues:", len(df_orders_delivered))
print("Linhas após integração:", len(df_vendas))

print(
    "Pedidos distintos após integração:",
    df_vendas["order_id"].nunique()
)

print(
    "Pedidos entregues sem itens:",
    len(df_orders_delivered)
    - df_vendas["order_id"].nunique()
)

display(df_vendas.head())



### 5.2 Integração das Bases de Pedidos e Itens

Foi realizada a integração entre a base de pedidos entregues (`df_orders_delivered`) e a base de itens (`df_items_clean`), utilizando a função `pd.merge()` da biblioteca Pandas.

A chave utilizada foi `order_id`, com relacionamento do tipo `one_to_many`, pois um pedido pode conter vários itens.

#### Resultados da integração

| Indicador | Quantidade |
|---|---:|
| Pedidos entregues | 96.478 |
| Linhas após integração | 110.197 |
| Pedidos distintos | 96.478 |
| Pedidos sem itens correspondentes | 0 |

#### Interpretação

A integração resultou em 110.197 registros de itens associados a 96.478 pedidos entregues.

A diferença entre a quantidade de pedidos e de linhas ocorre porque alguns pedidos possuem múltiplos itens.

Todos os pedidos entregues foram preservados na integração.

#### Validação

A utilização do parâmetro `validate="one_to_many"` confirmou a estrutura esperada do relacionamento.

Não foram identificados pedidos entregues sem itens correspondentes.

#### Conclusão

A integração foi concluída com sucesso, preservando os pedidos entregues e permitindo analisar os valores de cada item comercializado.

A base integrada será utilizada nas próximas etapas para acrescentar informações dos produtos e clientes.


In [ ]:

# Integrar a base de produtos

# Guardar a quantidade de linhas antes da integração
linhas_antes = len(df_vendas)

# Integrar os produtos
df_vendas = pd.merge(
    df_vendas,
    df_products_clean,
    on="product_id",
    how="left",
    validate="many_to_one"
)

print("===== INTEGRAÇÃO COM PRODUTOS =====")

print("Linhas antes da integração:", linhas_antes)

print("Linhas depois da integração:", len(df_vendas))

print(
    "Diferença de linhas:",
    len(df_vendas) - linhas_antes
)

print(
    "Categorias não informadas:",
    (
        df_vendas["product_category_name"]
        == "categoria_nao_informada"
    ).sum()
)

print(
    "Produtos sem correspondência:",
    df_vendas["product_weight_g"].isna().sum()
)

display(
    df_vendas[
        [
            "order_id",
            "product_id",
            "product_category_name",
            "price",
            "freight_value"
        ]
    ].head(10)
)



### 5.3 Integração da Base de Produtos

Foi realizada a integração da base de vendas com a base de produtos previamente tratada (`df_products_clean`).

A operação utilizou a função `pd.merge()`, com a chave `product_id` e relacionamento do tipo `many_to_one`.

#### Resultados da integração

| Indicador | Resultado |
|---|---:|
| Linhas antes da integração | 110.197 |
| Linhas depois da integração | 110.197 |
| Diferença de linhas | 0 |
| Itens com categoria não informada | 1.537 |
| Itens com peso ausente | 18 |

#### Interpretação

A integração preservou a quantidade de linhas da base de vendas, sem multiplicação ou perda de registros.

Foram identificados 1.537 itens associados a produtos cuja categoria original não estava informada.

A diferença em relação aos 610 produtos sem categoria identificados anteriormente ocorre porque um mesmo produto pode aparecer em vários itens vendidos.

Também foram identificados 18 itens associados a produtos com peso ausente.

Esses registros não representam necessariamente ausência de correspondência entre as bases.

#### Decisão

Os registros foram preservados, incluindo aqueles com categoria não informada ou peso ausente.

#### Conclusão

A integração permitiu acrescentar informações cadastrais dos produtos à base de vendas.

A base está preparada para receber os dados geográficos dos clientes e, posteriormente, realizar análises de vendas por categoria.


In [ ]:

# Integrar a base de clientes

# Guardar quantidade de linhas antes da integração
linhas_antes = len(df_vendas)

# Integrar os clientes
df_vendas = pd.merge(
    df_vendas,
    df_customers_clean,
    on="customer_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("===== INTEGRAÇÃO COM CLIENTES =====")

print("Linhas antes:", linhas_antes)
print("Linhas depois:", len(df_vendas))

print(
    "Diferença de linhas:",
    len(df_vendas) - linhas_antes
)

print("\n===== VALIDAÇÃO DO RELACIONAMENTO =====")

print(df_vendas["_merge"].value_counts())

print(
    "\nEstados ausentes:",
    df_vendas["customer_state"].isna().sum()
)

print(
    "Pedidos distintos:",
    df_vendas["order_id"].nunique()
)

# Remover coluna auxiliar de validação
df_vendas = df_vendas.drop(columns="_merge")

# Visualizar resultado
display(
    df_vendas[
        [
            "order_id",
            "customer_id",
            "customer_city",
            "customer_state",
            "product_category_name",
            "price",
            "freight_value"
        ]
    ].head(10)
)



### 5.4 Integração da Base de Clientes

Foi realizada a integração da base de vendas com a base de clientes (`df_customers_clean`), utilizando a função `pd.merge()` e a chave `customer_id`.

O relacionamento foi definido como `many_to_one`, pois um cliente associado a um pedido pode aparecer em várias linhas de itens vendidos.

#### Resultados da integração

| Indicador | Quantidade |
|---|---:|
| Linhas antes da integração | 110.197 |
| Linhas depois da integração | 110.197 |
| Diferença de linhas | 0 |
| Registros com correspondência | 110.197 |
| Registros sem correspondência | 0 |
| Estados ausentes | 0 |
| Pedidos distintos | 96.478 |

#### Validação

Foi utilizado o parâmetro `indicator=True` para verificar a correspondência entre as bases.

Todos os 110.197 registros apresentaram o indicador `both`, confirmando que encontraram clientes correspondentes.

Não houve aumento ou redução da quantidade de linhas.

#### Interpretação

A integração permitiu acrescentar informações geográficas dos clientes, incluindo cidade e estado, aos itens dos pedidos entregues.

Essas informações serão utilizadas para investigar a distribuição geográfica do valor das vendas.

#### Conclusão

A integração das quatro bases foi concluída com sucesso.

A base analítica contém 110.197 registros de itens associados a 96.478 pedidos entregues.

Os relacionamentos foram validados, permitindo avançar para a criação das variáveis necessárias à análise exploratória.


In [ ]:

# Criação das variáveis de vendas

# Criar cópia da base integrada
df_vendas_analise = df_vendas.copy()

# Valor comercial do item, sem frete
df_vendas_analise["valor_venda_item"] = (
    df_vendas_analise["price"]
)

# Valor do item incluindo frete
df_vendas_analise["valor_total_item"] = (
    df_vendas_analise["price"]
    + df_vendas_analise["freight_value"]
)

# Extrair ano e mês da compra
df_vendas_analise["ano_compra"] = (
    df_vendas_analise["order_purchase_timestamp"].dt.year
)

df_vendas_analise["mes_compra"] = (
    df_vendas_analise["order_purchase_timestamp"].dt.month
)

# Criar período mensal
df_vendas_analise["ano_mes"] = (
    df_vendas_analise["order_purchase_timestamp"]
    .dt.to_period("M")
)

print("===== VARIÁVEIS CRIADAS =====")

display(
    df_vendas_analise[
        [
            "order_id",
            "order_purchase_timestamp",
            "ano_compra",
            "mes_compra",
            "ano_mes",
            "customer_state",
            "product_category_name",
            "valor_venda_item",
            "freight_value",
            "valor_total_item"
        ]
    ].head(10)
)

print("\nTotal de registros:", len(df_vendas_analise))

print(
    "Pedidos distintos:",
    df_vendas_analise["order_id"].nunique()
)



### 5.5 Criação das Variáveis de Análise

Após a integração das quatro bases, foram criadas variáveis derivadas para permitir a análise temporal, geográfica e comercial das vendas.

#### Variáveis criadas

| Variável | Descrição |
|---|---|
| `valor_venda_item` | Preço de cada item vendido, sem frete |
| `valor_total_item` | Preço do item acrescido do frete |
| `ano_compra` | Ano da realização da compra |
| `mes_compra` | Mês da realização da compra |
| `ano_mes` | Período mensal da compra |

#### Procedimentos utilizados

Foram utilizadas operações aritméticas e funcionalidades de datas da biblioteca Pandas.

A variável `valor_venda_item` foi criada a partir de `price`, enquanto `valor_total_item` corresponde à soma de `price` e `freight_value`.

As variáveis temporais foram extraídas de `order_purchase_timestamp`, utilizando os atributos `.dt.year`, `.dt.month` e `.dt.to_period("M")`.

#### Validação

- Registros de itens: 110.197.
- Pedidos distintos: 96.478.
- Nenhum registro foi removido durante a criação das variáveis.

#### Justificativa

A separação entre o preço dos produtos e o valor do frete permite analisar o valor comercializado sem incluir os custos de transporte.

A criação de variáveis temporais possibilita investigar a evolução mensal das vendas.

#### Conclusão

As variáveis foram criadas com sucesso, mantendo a estrutura da base integrada e preparando os dados para as análises exploratórias.


In [ ]:

# Calcular valor total por pedido

df_pedidos_valores = (
    df_vendas_analise
    .groupby("order_id", as_index=False)
    .agg(
        valor_pedido=("valor_venda_item", "sum"),
        frete_pedido=("freight_value", "sum"),
        quantidade_itens=("order_item_id", "count"),
        data_compra=("order_purchase_timestamp", "first"),
        estado_cliente=("customer_state", "first"),
        ano_mes=("ano_mes", "first")
    )
)

# Valor total incluindo frete
df_pedidos_valores["valor_total_pedido"] = (
    df_pedidos_valores["valor_pedido"]
    + df_pedidos_valores["frete_pedido"]
)

print("===== VALORES POR PEDIDO =====")

print(
    "Total de pedidos:",
    len(df_pedidos_valores)
)

print(
    "Pedidos distintos:",
    df_pedidos_valores["order_id"].nunique()
)

print(
    "Valor total das vendas (sem frete): R$",
    round(df_pedidos_valores["valor_pedido"].sum(), 2)
)

print(
    "Valor total dos fretes: R$",
    round(df_pedidos_valores["frete_pedido"].sum(), 2)
)

print(
    "Ticket médio por pedido: R$",
    round(df_pedidos_valores["valor_pedido"].mean(), 2)
)

display(df_pedidos_valores.head(10))



### 5.6 Agregação dos Valores por Pedido

Após a criação das variáveis comerciais, foi realizada a agregação dos itens por pedido, utilizando o método `groupby()` da biblioteca Pandas.

O objetivo foi construir uma base com uma linha por pedido entregue, evitando a contagem duplicada de pedidos que possuem múltiplos itens.

#### Variáveis calculadas

- `valor_pedido`: soma dos preços dos itens do pedido, sem frete.
- `frete_pedido`: soma dos valores de frete dos itens.
- `quantidade_itens`: quantidade de itens registrados no pedido.
- `valor_total_pedido`: soma do valor dos produtos e dos fretes.

Também foram preservadas a data da compra, o estado do cliente e o período mensal.

#### Resultados obtidos

| Indicador | Resultado |
|---|---:|
| Pedidos entregues | 96.478 |
| Pedidos distintos | 96.478 |
| Valor total das vendas sem frete | R$ 13.221.498,11 |
| Valor total dos fretes | R$ 2.198.275,64 |
| Ticket médio por pedido sem frete | R$ 137,04 |

#### Interpretação

O valor total comercializado em produtos nos pedidos entregues foi de aproximadamente R$ 13,22 milhões, sem incluir os fretes.

O ticket médio foi de R$ 137,04 por pedido.

A quantidade de registros após a agregação corresponde exatamente à quantidade de pedidos distintos, confirmando que cada pedido passou a ocupar uma única linha.

#### Justificativa

A agregação por `order_id` evita que pedidos com múltiplos itens sejam contabilizados mais de uma vez em análises de quantidade de pedidos e ticket médio.

#### Conclusão

Foi criada a base `df_pedidos_valores`, contendo os valores comerciais consolidados por pedido.

Essa base será utilizada nas análises temporais, geográficas e de distribuição dos valores dos pedidos.


In [ ]:

# Validar valores antes e depois da agregação

# Totais antes da agregação
vendas_antes = df_vendas_analise["valor_venda_item"].sum()
fretes_antes = df_vendas_analise["freight_value"].sum()

# Totais depois da agregação
vendas_depois = df_pedidos_valores["valor_pedido"].sum()
fretes_depois = df_pedidos_valores["frete_pedido"].sum()

# Criar tabela comparativa
validacao_valores = pd.DataFrame({
    "indicador": [
        "Valor das vendas",
        "Valor dos fretes"
    ],
    "antes": [
        vendas_antes,
        fretes_antes
    ],
    "depois": [
        vendas_depois,
        fretes_depois
    ]
})

# Calcular diferenças
validacao_valores["diferenca"] = (
    validacao_valores["depois"]
    - validacao_valores["antes"]
)

# Verificar igualdade com tolerância numérica
import numpy as np

validacao_valores["valores_conferem"] = np.isclose(
    validacao_valores["antes"],
    validacao_valores["depois"],
    rtol=0,
    atol=0.01
)

print("===== VALIDAÇÃO DOS VALORES =====")

display(validacao_valores.round(2))

print(
    "\nTodos os valores conferem:",
    validacao_valores["valores_conferem"].all()
)



### 5.7 Validação dos Valores Após a Agregação

Após o agrupamento dos itens por pedido, foi realizada uma comparação dos valores comerciais antes e depois da transformação.

O objetivo foi verificar se a agregação preservou integralmente os valores das vendas e dos fretes.

#### Resultados

| Indicador | Antes | Depois |
|---|---:|---:|
| Valor das vendas sem frete | R$ 13.221.498,11 | R$ 13.221.498,11 |
| Valor dos fretes | R$ 2.198.275,64 | R$ 2.198.275,64 |

A verificação foi realizada utilizando a função `np.isclose()`, com tolerância absoluta de R$ 0,01.

O resultado foi `True` para ambos os indicadores.

#### Interpretação

Os valores totais permaneceram equivalentes após a agregação dos itens por pedido.

A pequena diferença numérica apresentada como `-0.00` decorre da precisão dos cálculos com números de ponto flutuante.

#### Conclusão

A transformação preservou os valores comerciais da base.

A validação confirma que a agregação não provocou perda ou duplicação de valores, permitindo utilizar a base consolidada nas próximas análises.


In [ ]:

# Preparação das vendas mensais

df_vendas_mensais = (
    df_pedidos_valores
    .groupby("ano_mes", as_index=False)
    .agg(
        valor_vendas=("valor_pedido", "sum"),
        quantidade_pedidos=("order_id", "count"),
        ticket_medio=("valor_pedido", "mean")
    )
    .sort_values("ano_mes")
    .reset_index(drop=True)
)

# Criar variação percentual mensal
df_vendas_mensais["variacao_mensal_pct"] = (
    df_vendas_mensais["valor_vendas"]
    .pct_change() * 100
)

print("===== EVOLUÇÃO MENSAL DAS VENDAS =====")

display(df_vendas_mensais)

print(
    "\nQuantidade de meses analisados:",
    len(df_vendas_mensais)
)

print(
    "Total das vendas mensais: R$",
    round(df_vendas_mensais["valor_vendas"].sum(), 2)
)

print(
    "Total de pedidos:",
    df_vendas_mensais["quantidade_pedidos"].sum()
)



### 5.8 Preparação da Evolução Mensal das Vendas

Foi realizada a agregação dos pedidos entregues por mês de compra, utilizando o método `groupby()` da biblioteca Pandas.

O objetivo foi preparar os dados para investigar a evolução temporal das vendas da Olist.

#### Variáveis calculadas

- `valor_vendas`: soma dos valores dos produtos vendidos no mês, sem frete.
- `quantidade_pedidos`: quantidade de pedidos entregues associados ao mês de compra.
- `ticket_medio`: valor médio dos pedidos no mês.
- `variacao_mensal_pct`: variação percentual das vendas em relação ao período anterior disponível.

#### Resultados

- Períodos mensais com registros: 23.
- Total das vendas: R$ 13.221.498,11.
- Total de pedidos entregues: 96.478.
- Maior valor mensal: novembro de 2017, com R$ 987.765,37.

#### Interpretação

Os dados apresentam crescimento expressivo das vendas ao longo de 2017, com oscilações entre os meses.

Em 2018, os valores mensais permaneceram em patamares elevados entre janeiro e agosto.

Os primeiros meses da base apresentam baixa quantidade de pedidos, produzindo variações percentuais extremas que não devem ser interpretadas isoladamente.

#### Limitações

O conjunto de dados não apresenta cobertura uniforme em todo o período.

Novembro de 2016 não possui registros na tabela mensal, e os meses iniciais apresentam quantidades reduzidas de pedidos.

A variação percentual foi calculada entre períodos disponíveis, que nem sempre são meses consecutivos.

#### Conclusão

A base `df_vendas_mensais` foi preparada para a análise temporal, preservando o total de vendas e a quantidade de pedidos.

A interpretação dos resultados deverá considerar as limitações de cobertura temporal.


In [ ]:

# Preparação das vendas por estado

df_vendas_estados = (
    df_pedidos_valores
    .groupby("estado_cliente", as_index=False)
    .agg(
        valor_vendas=("valor_pedido", "sum"),
        quantidade_pedidos=("order_id", "count"),
        ticket_medio=("valor_pedido", "mean")
    )
    .sort_values("valor_vendas", ascending=False)
    .reset_index(drop=True)
)

# Calcular participação percentual nas vendas
total_vendas = df_vendas_estados["valor_vendas"].sum()

df_vendas_estados["participacao_pct"] = (
    df_vendas_estados["valor_vendas"]
    / total_vendas * 100
)

print("===== VENDAS POR ESTADO =====")

display(df_vendas_estados.round(2))

print(
    "\nQuantidade de estados:",
    df_vendas_estados["estado_cliente"].nunique()
)

print(
    "Total das vendas: R$",
    round(df_vendas_estados["valor_vendas"].sum(), 2)
)

print(
    "Total de pedidos:",
    df_vendas_estados["quantidade_pedidos"].sum()
)

print("\n===== 5 ESTADOS COM MAIOR VALOR DE VENDAS =====")

display(df_vendas_estados.head(5).round(2))



### 5.9 Preparação das Vendas por Estado

Foi realizada a agregação dos pedidos entregues por estado do cliente, utilizando o método `groupby()` da biblioteca Pandas.

O objetivo foi identificar a distribuição geográfica das vendas e os estados com maior participação no valor total comercializado.

#### Variáveis calculadas

- `valor_vendas`: soma dos valores dos produtos vendidos em cada estado, sem frete.
- `quantidade_pedidos`: quantidade de pedidos entregues por estado.
- `ticket_medio`: valor médio dos pedidos por estado.
- `participacao_pct`: participação percentual de cada estado no valor total das vendas.

#### Resultados

Foram identificadas as 27 unidades federativas brasileiras.

O valor total das vendas foi de R$ 13.221.498,11, distribuído entre 96.478 pedidos entregues.

#### Cinco estados com maior valor de vendas

| Estado | Valor das vendas | Participação |
|---|---:|---:|
| SP | R$ 5.067.633,16 | 38,33% |
| RJ | R$ 1.759.651,13 | 13,31% |
| MG | R$ 1.552.481,83 | 11,74% |
| RS | R$ 728.897,47 | 5,51% |
| PR | R$ 666.063,51 | 5,04% |

#### Interpretação

São Paulo apresentou o maior valor de vendas, representando 38,33% do total comercializado.

Os estados de São Paulo, Rio de Janeiro e Minas Gerais concentraram aproximadamente 63,38% das vendas.

Os cinco primeiros estados representaram aproximadamente 73,93% do valor total.

Essa distribuição demonstra uma concentração expressiva das vendas na região Sudeste.

Entretanto, o maior valor total de vendas não significa necessariamente maior ticket médio.

A Paraíba, por exemplo, apresentou ticket médio de R$ 217,77, apesar de possuir participação relativamente pequena no valor total comercializado.

#### Conclusão

A base `df_vendas_estados` foi preparada para investigar a distribuição geográfica das vendas.

Os resultados indicam concentração do valor comercializado em poucos estados, especialmente São Paulo, Rio de Janeiro e Minas Gerais.


In [ ]:

# Preparação das vendas por categoria

df_vendas_categorias = (
    df_vendas_analise
    .groupby("product_category_name", as_index=False)
    .agg(
        valor_vendas=("valor_venda_item", "sum"),
        quantidade_itens=("order_item_id", "count"),
        quantidade_pedidos=("order_id", "nunique"),
        preco_medio=("valor_venda_item", "mean")
    )
    .sort_values("valor_vendas", ascending=False)
    .reset_index(drop=True)
)

# Participação percentual de cada categoria
total_vendas = df_vendas_categorias["valor_vendas"].sum()

df_vendas_categorias["participacao_pct"] = (
    df_vendas_categorias["valor_vendas"]
    / total_vendas * 100
)

print("===== VENDAS POR CATEGORIA =====")

display(df_vendas_categorias.round(2))

print(
    "\nQuantidade de categorias:",
    df_vendas_categorias["product_category_name"].nunique()
)

print(
    "Total das vendas: R$",
    round(df_vendas_categorias["valor_vendas"].sum(), 2)
)

print(
    "Total de itens:",
    df_vendas_categorias["quantidade_itens"].sum()
)

print("\n===== 10 CATEGORIAS COM MAIOR VALOR DE VENDAS =====")

display(df_vendas_categorias.head(10).round(2))



### 5.10 Preparação das Vendas por Categoria

Foi realizada a agregação dos itens vendidos por categoria de produto, utilizando o método `groupby()` da biblioteca Pandas.

O objetivo foi identificar as categorias com maior valor comercializado e avaliar sua participação nas vendas totais.

#### Variáveis calculadas

- `valor_vendas`: soma dos preços dos itens por categoria, sem frete.
- `quantidade_itens`: quantidade de itens vendidos por categoria.
- `quantidade_pedidos`: número de pedidos distintos associados à categoria.
- `preco_medio`: preço médio dos itens da categoria.
- `participacao_pct`: participação percentual no valor total das vendas.

#### Resultados gerais

- Categorias identificadas: 74, incluindo `categoria_nao_informada`.
- Valor total das vendas: R$ 13.221.498,11.
- Quantidade de itens: 110.197.

#### Cinco categorias com maior valor de vendas

| Categoria | Valor das vendas | Participação |
|---|---:|---:|
| Beleza e saúde | R$ 1.233.131,72 | 9,33% |
| Relógios e presentes | R$ 1.166.176,98 | 8,82% |
| Cama, mesa e banho | R$ 1.023.434,76 | 7,74% |
| Esporte e lazer | R$ 954.852,55 | 7,22% |
| Informática e acessórios | R$ 888.724,61 | 6,72% |

#### Interpretação

A categoria beleza e saúde apresentou o maior valor comercializado, seguida por relógios e presentes e cama, mesa e banho.

As cinco categorias líderes concentraram aproximadamente 39,83% das vendas totais.

A categoria cama, mesa e banho apresentou maior quantidade de itens vendidos entre as cinco primeiras, mas não liderou em valor financeiro.

Isso demonstra que o valor total comercializado depende tanto da quantidade de itens quanto dos preços praticados.

#### Limitações

Um pedido pode conter itens de diferentes categorias. Portanto, a soma dos pedidos distintos por categoria pode ultrapassar a quantidade total de pedidos da base.

Os produtos sem categoria identificada foram preservados como `categoria_nao_informada`.

#### Conclusão

A base `df_vendas_categorias` foi preparada para identificar as categorias com maior participação nas vendas e comparar seus resultados comerciais.


In [ ]:

# Distribuição dos valores dos pedidos

print("===== ESTATÍSTICAS DOS VALORES DOS PEDIDOS =====")

estatisticas_pedidos = (
    df_pedidos_valores["valor_pedido"]
    .describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

display(estatisticas_pedidos.round(2))

# Calcular assimetria
assimetria = df_pedidos_valores["valor_pedido"].skew()

print("\nAssimetria:", round(assimetria, 2))

# Identificar possíveis outliers pelo método IQR
q1 = df_pedidos_valores["valor_pedido"].quantile(0.25)
q3 = df_pedidos_valores["valor_pedido"].quantile(0.75)

iqr = q3 - q1

limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

outliers = (
    (df_pedidos_valores["valor_pedido"] < limite_inferior)
    |
    (df_pedidos_valores["valor_pedido"] > limite_superior)
)

print("\n===== POSSÍVEIS OUTLIERS =====")

print("Q1: R$", round(q1, 2))
print("Q3: R$", round(q3, 2))
print("IQR: R$", round(iqr, 2))
print("Limite inferior: R$", round(limite_inferior, 2))
print("Limite superior: R$", round(limite_superior, 2))

print("Quantidade de possíveis outliers:", outliers.sum())

print(
    "Percentual de possíveis outliers:",
    round(outliers.mean() * 100, 2),
    "%"
)



### 5.11 Análise Descritiva da Distribuição dos Valores dos Pedidos

Foi realizada uma análise estatística dos valores dos pedidos entregues, utilizando medidas de tendência central, dispersão, posição e assimetria.

O objetivo foi compreender a distribuição dos valores comerciais e identificar possíveis valores extremos.

#### Estatísticas descritivas

| Indicador | Resultado |
|---|---:|
| Quantidade de pedidos | 96.478 |
| Média | R$ 137,04 |
| Mediana | R$ 86,57 |
| Desvio-padrão | R$ 209,05 |
| Primeiro quartil | R$ 45,90 |
| Terceiro quartil | R$ 149,90 |
| Percentil 90 | R$ 269,00 |
| Percentil 95 | R$ 399,00 |
| Percentil 99 | R$ 990,00 |
| Valor máximo | R$ 13.440,00 |
| Assimetria | 9,89 |

#### Interpretação

A média dos pedidos (R$ 137,04) é superior à mediana (R$ 86,57), indicando influência dos valores mais elevados sobre a média.

O coeficiente de assimetria de 9,89 confirma uma distribuição fortemente assimétrica à direita.

Aproximadamente 75% dos pedidos apresentam valores de até R$ 149,90.

#### Identificação de possíveis outliers

Foi aplicado o método do intervalo interquartil (IQR).

- Q1: R$ 45,90.
- Q3: R$ 149,90.
- IQR: R$ 104,00.
- Limite inferior: R$ -110,10.
- Limite superior: R$ 305,90.
- Possíveis outliers: 7.659 pedidos.
- Percentual de possíveis outliers: 7,94%.

#### Decisão de tratamento

Os valores extremos serão preservados.

#### Justificativa

Pedidos de valores elevados podem representar transações comerciais legítimas.

A classificação estatística como outlier não é suficiente para concluir que existe erro no registro.

A exclusão desses pedidos poderia distorcer o valor total comercializado e o comportamento real da distribuição.

#### Conclusão

A distribuição dos valores dos pedidos apresenta forte assimetria positiva, com concentração de pedidos em valores relativamente baixos e presença de uma cauda de valores elevados.

Essas características deverão ser consideradas na interpretação dos gráficos da análise exploratória.


In [ ]:

# Validação final da Etapa 5

import numpy as np

# Valores de referência
total_vendas = df_vendas_analise["valor_venda_item"].sum()
total_pedidos = df_orders_delivered["order_id"].nunique()
total_itens = len(df_vendas_analise)

# Verificações
validacoes = [
    {
        "verificacao": "Pedidos após agregação",
        "resultado": len(df_pedidos_valores),
        "esperado": total_pedidos,
        "confere": len(df_pedidos_valores) == total_pedidos
    },
    {
        "verificacao": "Itens após integração",
        "resultado": len(df_vendas_analise),
        "esperado": total_itens,
        "confere": len(df_vendas_analise) == total_itens
    },
    {
        "verificacao": "Vendas por pedido",
        "resultado": df_pedidos_valores["valor_pedido"].sum(),
        "esperado": total_vendas,
        "confere": np.isclose(
            df_pedidos_valores["valor_pedido"].sum(),
            total_vendas,
            rtol=0,
            atol=0.01
        )
    },
    {
        "verificacao": "Vendas mensais",
        "resultado": df_vendas_mensais["valor_vendas"].sum(),
        "esperado": total_vendas,
        "confere": np.isclose(
            df_vendas_mensais["valor_vendas"].sum(),
            total_vendas,
            rtol=0,
            atol=0.01
        )
    },
    {
        "verificacao": "Vendas por estado",
        "resultado": df_vendas_estados["valor_vendas"].sum(),
        "esperado": total_vendas,
        "confere": np.isclose(
            df_vendas_estados["valor_vendas"].sum(),
            total_vendas,
            rtol=0,
            atol=0.01
        )
    },
    {
        "verificacao": "Vendas por categoria",
        "resultado": df_vendas_categorias["valor_vendas"].sum(),
        "esperado": total_vendas,
        "confere": np.isclose(
            df_vendas_categorias["valor_vendas"].sum(),
            total_vendas,
            rtol=0,
            atol=0.01
        )
    }
]

df_validacao_final = pd.DataFrame(validacoes)

print("===== VALIDAÇÃO FINAL DA ETAPA 5 =====")

display(df_validacao_final.round(2))

print(
    "\nTodas as verificações conferem:",
    df_validacao_final["confere"].all()
)



### 5.12 Validação Final das Bases Analíticas

Após a integração, transformação e agregação dos dados, foi realizada uma validação final para verificar a consistência das bases preparadas.

#### Objetivo

Confirmar que os procedimentos de preparação não provocaram perdas ou duplicações nos totais de pedidos, itens e valores comerciais.

#### Resultados da validação

| Verificação | Resultado | Situação |
|---|---:|---|
| Pedidos após agregação | 96.478 | Conforme |
| Itens após integração | 110.197 | Conforme |
| Vendas por pedido | R$ 13.221.498,11 | Conforme |
| Vendas mensais | R$ 13.221.498,11 | Conforme |
| Vendas por estado | R$ 13.221.498,11 | Conforme |
| Vendas por categoria | R$ 13.221.498,11 | Conforme |

Todas as seis verificações apresentaram resultado `True`.

#### Interpretação

Os resultados demonstram que as agregações realizadas preservaram os totais comerciais da base analisada.

As diferentes perspectivas de análise apresentam valores consistentes, permitindo a comparação dos resultados temporais, geográficos e comerciais.

#### Conclusão da Etapa 5

A preparação dos dados foi concluída, contemplando:

1. Integração das bases de pedidos, itens, produtos e clientes.
2. Criação de variáveis temporais e comerciais.
3. Agregação dos valores por pedido.
4. Preparação das vendas mensais.
5. Preparação das vendas por estado.
6. Preparação das vendas por categoria.
7. Análise descritiva dos valores dos pedidos.
8. Validação final dos totais comerciais.

As bases estão preparadas para a etapa de análise exploratória e visualização dos dados.



## Etapa 6 — Análise Exploratória e Visualização dos Dados

Nesta etapa, serão utilizadas técnicas de visualização de dados para investigar padrões temporais, geográficos e comerciais nas vendas da Olist.

### Objetivo

Responder às perguntas de pesquisa definidas no início do projeto por meio de gráficos e análises estatísticas.

### Análises propostas

1. Evolução mensal do valor das vendas.
2. Distribuição das vendas por estado.
3. Distribuição dos valores dos pedidos.
4. Categorias de produtos com maior valor de vendas.

As visualizações serão construídas com Python, utilizando as bibliotecas Pandas, Matplotlib e Seaborn.

Os resultados serão interpretados considerando as limitações identificadas durante o diagnóstico e a limpeza dos dados.


In [ ]:

# Evolução mensal das vendas

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.dates as mdates

# Preparar os dados
df_grafico = df_vendas_mensais.copy()

# Converter ano_mes para data
df_grafico["data"] = pd.to_datetime(
    df_grafico["ano_mes"].astype(str)
)

# Selecionar o período principal da análise
df_grafico = df_grafico[
    (df_grafico["data"] >= "2017-01-01") &
    (df_grafico["data"] <= "2018-08-31")
].copy()

# Criar gráfico
fig, ax = plt.subplots(figsize=(13, 6))

ax.plot(
    df_grafico["data"],
    df_grafico["valor_vendas"],
    color="#4285F4",
    marker="o",
    linewidth=2.5,
    markersize=6,
    label="Vendas"
)

# Título
ax.set_title(
    "Evolução Mensal das Vendas - Olist",
    fontsize=16,
    fontweight="bold",
    pad=20
)

# Eixos
ax.set_xlabel("Mês da compra", fontsize=11)
ax.set_ylabel("Valor das vendas (R$)", fontsize=11)

# Formatação monetária
ax.yaxis.set_major_formatter(
    ticker.FuncFormatter(
        lambda x, pos: f"R$ {x/1_000_000:.1f} mi"
        if x >= 1_000_000
        else f"R$ {x/1000:.0f} mil"
    )
)

# Formatar datas
ax.xaxis.set_major_locator(
    mdates.MonthLocator(interval=2)
)

ax.xaxis.set_major_formatter(
    mdates.DateFormatter("%m/%Y")
)

plt.xticks(rotation=45)

# Grade
ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.3
)

# Remover bordas desnecessárias
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Legenda
ax.legend(frameon=False)

# Ajustar layout
plt.tight_layout()

# Exibir
plt.show()



### 6.1.1 Interpretação da Evolução Mensal das Vendas

O gráfico de linhas apresenta a evolução mensal do valor das vendas da Olist entre janeiro de 2017 e agosto de 2018, considerando apenas os pedidos entregues e os valores dos produtos, sem frete.

#### Principais resultados

1. **Crescimento das vendas em 2017:** observa-se uma tendência de crescimento ao longo do ano, apesar de oscilações mensais.

2. **Maior valor mensal:** novembro de 2017 apresentou o maior valor de vendas, totalizando R$ 987.765,37.

3. **Comportamento em 2018:** entre janeiro e agosto de 2018, as vendas permaneceram em patamares superiores a R$ 800 mil mensais, apresentando oscilações.

4. **Variações temporais:** a evolução não ocorreu de maneira linear, demonstrando períodos de crescimento e redução no valor comercializado.

#### Limitações

A análise considera apenas pedidos com status de entrega concluída.

Os resultados representam os valores dos produtos associados ao mês da compra, não incluindo os fretes.

O período visualizado foi delimitado entre janeiro de 2017 e agosto de 2018 para evitar distorções decorrentes da baixa quantidade de registros nos primeiros meses da base.

#### Conclusão

Os resultados indicam crescimento expressivo das vendas ao longo de 2017, seguido por oscilações em níveis mais elevados durante 2018.

O gráfico responde à primeira pergunta auxiliar do projeto, demonstrando a evolução temporal das vendas da Olist.


In [ ]:

# Gráfico das vendas por estado

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Selecionar os 10 estados com maiores vendas
top_10_estados = (
    df_vendas_estados
    .nlargest(10, "valor_vendas")
    .sort_values("valor_vendas")
)

# Criar gráfico
fig, ax = plt.subplots(figsize=(12, 7))

barras = ax.barh(
    top_10_estados["estado_cliente"],
    top_10_estados["valor_vendas"],
    color="#4285F4"
)

# Título e eixos
ax.set_title(
    "10 Estados com Maior Valor de Vendas - Olist",
    fontsize=16,
    fontweight="bold",
    pad=20
)

ax.set_xlabel("Valor das vendas (R$)", fontsize=11)
ax.set_ylabel("Estado do cliente", fontsize=11)

# Formatar valores monetários
ax.xaxis.set_major_formatter(
    ticker.FuncFormatter(
        lambda x, pos: f"R$ {x/1_000_000:.1f} mi"
    )
)

# Adicionar valores nas barras
for barra in barras:
    valor = barra.get_width()

    ax.text(
        valor + 50000,
        barra.get_y() + barra.get_height() / 2,
        f"R$ {valor/1_000_000:.2f} mi",
        va="center",
        fontsize=10
    )

# Ajustar limite para comportar os rótulos
ax.set_xlim(
    0,
    top_10_estados["valor_vendas"].max() * 1.25
)

# Grade vertical
ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.3
)

ax.set_axisbelow(True)

# Remover bordas
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()



### 6.2.1 Interpretação das Vendas por Estado

O gráfico de barras horizontais apresenta os dez estados brasileiros com maior valor de vendas na base Olist.

A análise considera os pedidos entregues e os valores dos produtos comercializados, sem incluir os fretes.

#### Principais resultados

1. **São Paulo:** apresentou o maior valor de vendas, aproximadamente R$ 5,07 milhões, representando 38,33% do total.

2. **Rio de Janeiro:** ocupou a segunda posição, com aproximadamente R$ 1,76 milhão e participação de 13,31%.

3. **Minas Gerais:** registrou aproximadamente R$ 1,55 milhão, correspondendo a 11,74% das vendas.

4. **Concentração geográfica:** os três estados com maior valor comercializado concentraram aproximadamente 63,38% das vendas totais.

5. **Demais estados:** Rio Grande do Sul e Paraná completaram as cinco primeiras posições, com valores inferiores aos registrados pelos três líderes.

#### Interpretação

A distribuição evidencia uma concentração expressiva das vendas na região Sudeste, especialmente no estado de São Paulo.

Entretanto, o maior valor comercializado não significa necessariamente maior ticket médio.

Além disso, os resultados representam a distribuição das vendas registradas na base, não sendo suficientes para explicar as causas das diferenças entre os estados.

#### Conclusão

A análise responde à segunda pergunta auxiliar do projeto, identificando os estados que concentram o maior valor de vendas da Olist.

São Paulo, Rio de Janeiro e Minas Gerais destacaram-se como os principais mercados consumidores no conjunto de dados analisado.


In [ ]:

# Distribuição dos valores dos pedidos

import matplotlib.pyplot as plt
import seaborn as sns

# Selecionar valores dos pedidos
valores = df_pedidos_valores["valor_pedido"]

# Calcular percentil 99
p99 = valores.quantile(0.99)

# Criar dois gráficos
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# GRÁFICO 1 - Distribuição completa
sns.histplot(
    valores,
    bins=60,
    color="#4285F4",
    ax=axes[0]
)

axes[0].set_title(
    "Distribuição Completa dos Pedidos",
    fontsize=13,
    fontweight="bold"
)

axes[0].set_xlabel("Valor do pedido (R$)")
axes[0].set_ylabel("Frequência")

# GRÁFICO 2 - Até o percentil 99
sns.histplot(
    valores[valores <= p99],
    bins=50,
    color="#34A853",
    ax=axes[1]
)

axes[1].set_title(
    "Distribuição dos Pedidos até o Percentil 99",
    fontsize=13,
    fontweight="bold"
)

axes[1].set_xlabel("Valor do pedido (R$)")
axes[1].set_ylabel("Frequência")

# Adicionar linha da mediana
axes[1].axvline(
    valores.median(),
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Mediana: R$ {valores.median():.2f}"
)

axes[1].legend()

# Melhorar aparência
for ax in axes:
    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.3
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.suptitle(
    "Análise da Distribuição dos Valores dos Pedidos - Olist",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()
plt.show()



### 6.3.1 Interpretação da Distribuição dos Valores dos Pedidos

Foram construídos dois histogramas para analisar a distribuição dos valores dos pedidos entregues da Olist.

O primeiro apresenta a distribuição completa dos pedidos, enquanto o segundo considera apenas os valores até o percentil 99, permitindo uma visualização mais detalhada da concentração dos valores.

#### Principais resultados

1. **Concentração em valores menores:** a maior parte dos pedidos está concentrada nas faixas de valores mais baixos.

2. **Assimetria positiva:** a distribuição apresenta uma cauda longa à direita, confirmada pelo coeficiente de assimetria de 9,89.

3. **Média e mediana:** o valor médio dos pedidos foi de R$ 137,04, enquanto a mediana foi de R$ 86,57.

4. **Valores extremos:** o maior pedido apresentou valor de R$ 13.440,00.

5. **Possíveis outliers:** pelo método IQR, foram identificados 7.659 pedidos acima do limite superior de R$ 305,90, correspondendo a 7,94% da base.

#### Interpretação estatística

A diferença entre média e mediana indica que pedidos de valores elevados influenciam a média.

A mediana representa melhor a tendência central da distribuição, pois é menos sensível aos valores extremos.

A concentração dos pedidos em valores menores e a presença de poucos pedidos muito elevados caracterizam uma distribuição fortemente assimétrica à direita.

#### Tratamento dos valores extremos

Os possíveis outliers foram preservados na base original, pois podem representar transações comerciais legítimas.

O segundo histograma utiliza um recorte visual até o percentil 99, sem alterar os dados utilizados nas demais análises.

#### Conclusão

A análise responde à terceira pergunta auxiliar do projeto, demonstrando que os valores dos pedidos não apresentam distribuição simétrica.

A maior parte dos pedidos possui valores relativamente baixos, enquanto uma parcela menor apresenta valores elevados que influenciam a média e a dispersão.


In [ ]:

# Gráfico das 10 categorias com maiores vendas

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Selecionar as 10 categorias com maiores vendas
top_10_categorias = (
    df_vendas_categorias
    .nlargest(10, "valor_vendas")
    .sort_values("valor_vendas")
)

# Melhorar os nomes para apresentação
nomes_categorias = (
    top_10_categorias["product_category_name"]
    .str.replace("_", " ", regex=False)
    .str.title()
)

# Criar gráfico
fig, ax = plt.subplots(figsize=(13, 7))

barras = ax.barh(
    nomes_categorias,
    top_10_categorias["valor_vendas"],
    color="#4285F4"
)

# Título
ax.set_title(
    "10 Categorias com Maior Valor de Vendas - Olist",
    fontsize=16,
    fontweight="bold",
    pad=20
)

# Eixos
ax.set_xlabel("Valor das vendas (R$)", fontsize=11)
ax.set_ylabel("Categoria do produto", fontsize=11)

# Formatar valores monetários
ax.xaxis.set_major_formatter(
    ticker.FuncFormatter(
        lambda x, pos: f"R$ {x/1_000_000:.1f} mi"
    )
)

# Adicionar valores nas barras
for barra in barras:
    valor = barra.get_width()

    ax.text(
        valor + 15000,
        barra.get_y() + barra.get_height() / 2,
        f"R$ {valor/1_000_000:.2f} mi",
        va="center",
        fontsize=10
    )

# Espaço para os rótulos
ax.set_xlim(
    0,
    top_10_categorias["valor_vendas"].max() * 1.25
)

# Grade
ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.3
)

ax.set_axisbelow(True)

# Remover bordas
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()



### 6.3.1 Interpretação da Distribuição dos Valores dos Pedidos

Foram construídos dois histogramas para analisar a distribuição dos valores dos pedidos entregues da Olist.

O primeiro apresenta a distribuição completa dos pedidos, enquanto o segundo considera apenas os valores até o percentil 99, permitindo uma visualização mais detalhada da concentração dos valores.

#### Principais resultados

1. **Concentração em valores menores:** a maior parte dos pedidos está concentrada nas faixas de valores mais baixos.

2. **Assimetria positiva:** a distribuição apresenta uma cauda longa à direita, confirmada pelo coeficiente de assimetria de 9,89.

3. **Média e mediana:** o valor médio dos pedidos foi de R$ 137,04, enquanto a mediana foi de R$ 86,57.

4. **Valores extremos:** o maior pedido apresentou valor de R$ 13.440,00.

5. **Possíveis outliers:** pelo método IQR, foram identificados 7.659 pedidos acima do limite superior de R$ 305,90, correspondendo a 7,94% da base.

#### Interpretação estatística

A diferença entre média e mediana indica que pedidos de valores elevados influenciam a média.

A mediana representa melhor a tendência central da distribuição, pois é menos sensível aos valores extremos.

A concentração dos pedidos em valores menores e a presença de poucos pedidos muito elevados caracterizam uma distribuição fortemente assimétrica à direita.

#### Tratamento dos valores extremos

Os possíveis outliers foram preservados na base original, pois podem representar transações comerciais legítimas.

O segundo histograma utiliza um recorte visual até o percentil 99, sem alterar os dados utilizados nas demais análises.

#### Conclusão

A análise responde à terceira pergunta auxiliar do projeto, demonstrando que os valores dos pedidos não apresentam distribuição simétrica.

A maior parte dos pedidos possui valores relativamente baixos, enquanto uma parcela menor apresenta valores elevados que influenciam a média e a dispersão.


In [ ]:

# Gráfico das 10 categorias com maiores vendas

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Selecionar as 10 categorias com maiores vendas
top_10_categorias = (
    df_vendas_categorias
    .nlargest(10, "valor_vendas")
    .sort_values("valor_vendas")
)

# Melhorar os nomes para apresentação
nomes_categorias = (
    top_10_categorias["product_category_name"]
    .str.replace("_", " ", regex=False)
    .str.title()
)

# Criar gráfico
fig, ax = plt.subplots(figsize=(13, 7))

barras = ax.barh(
    nomes_categorias,
    top_10_categorias["valor_vendas"],
    color="#4285F4"
)

# Título
ax.set_title(
    "10 Categorias com Maior Valor de Vendas - Olist",
    fontsize=16,
    fontweight="bold",
    pad=20
)

# Eixos
ax.set_xlabel("Valor das vendas (R$)", fontsize=11)
ax.set_ylabel("Categoria do produto", fontsize=11)

# Formatar valores monetários
ax.xaxis.set_major_formatter(
    ticker.FuncFormatter(
        lambda x, pos: f"R$ {x/1_000_000:.1f} mi"
    )
)

# Adicionar valores nas barras
for barra in barras:
    valor = barra.get_width()

    ax.text(
        valor + 15000,
        barra.get_y() + barra.get_height() / 2,
        f"R$ {valor/1_000_000:.2f} mi",
        va="center",
        fontsize=10
    )

# Espaço para os rótulos
ax.set_xlim(
    0,
    top_10_categorias["valor_vendas"].max() * 1.25
)

# Grade
ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.3
)

ax.set_axisbelow(True)

# Remover bordas
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()



### 6.4.1 Interpretação das Vendas por Categoria

O gráfico de barras horizontais apresenta as dez categorias de produtos com maior valor de vendas na base Olist.

A análise considera os valores dos produtos comercializados em pedidos entregues, sem incluir os fretes.

#### Principais resultados

1. **Beleza e saúde:** apresentou o maior valor de vendas, totalizando R$ 1.233.131,72, equivalente a 9,33% do total.

2. **Relógios e presentes:** ocupou a segunda posição, com R$ 1.166.176,98 e participação de 8,82%.

3. **Cama, mesa e banho:** registrou R$ 1.023.434,76, representando 7,74% das vendas.

4. **Esporte e lazer:** apresentou R$ 954.852,55, correspondendo a 7,22%.

5. **Informática e acessórios:** registrou R$ 888.724,61, equivalente a 6,72% das vendas.

#### Interpretação

As cinco categorias com maior valor comercializado concentraram aproximadamente 39,83% das vendas totais.

A categoria cama, mesa e banho apresentou maior quantidade de itens vendidos entre as cinco líderes, mas não ocupou a primeira posição em valor financeiro.

Esse resultado demonstra que o valor total comercializado depende tanto da quantidade de itens vendidos quanto dos preços dos produtos.

#### Limitações

A análise considera somente os pedidos entregues.

Os produtos sem categoria identificada foram preservados na base como `categoria_nao_informada`.

Os resultados representam os valores comercializados no conjunto de dados analisado e não permitem determinar as causas das diferenças entre as categorias.

#### Conclusão

A análise responde à quarta pergunta auxiliar do projeto, identificando as categorias com maior participação no valor total das vendas da Olist.

Beleza e saúde, relógios e presentes e cama, mesa e banho destacaram-se como as três categorias líderes em valor comercializado.



### 6.5 Síntese da Análise Exploratória

A análise exploratória permitiu identificar padrões temporais, geográficos e comerciais nas vendas da Olist.

#### Principais descobertas

- **Evolução temporal:** as vendas apresentaram crescimento expressivo ao longo de 2017, seguido por oscilações em patamares mais elevados durante 2018.

- **Distribuição geográfica:** São Paulo, Rio de Janeiro e Minas Gerais concentraram aproximadamente 63,38% do valor total das vendas.

- **Valores dos pedidos:** a distribuição apresentou forte assimetria positiva, com média de R$ 137,04 e mediana de R$ 86,57.

- **Categorias de produtos:** beleza e saúde liderou em valor comercializado, seguida por relógios e presentes e cama, mesa e banho.

#### Considerações

Os resultados foram obtidos a partir de pedidos entregues, considerando os valores dos produtos sem frete.

As decisões de limpeza preservaram registros potencialmente legítimos, incluindo valores extremos e produtos com categorias não informadas.

As análises são descritivas e não permitem estabelecer relações de causa e efeito.

#### Conclusão da Etapa 6

As quatro perguntas auxiliares foram respondidas por meio de estatísticas descritivas e visualizações.

Os resultados fornecem evidências para a elaboração das conclusões finais do projeto.



---

## 7. Conclusões, Limitações e Recomendações

### 7.1 Objetivo da etapa

Esta etapa apresenta as principais conclusões da Análise Exploratória de Dados (EDA) realizada com a base pública de e-commerce brasileiro Olist.

O objetivo é responder à pergunta principal do projeto, consolidar os resultados das análises e discutir as limitações dos dados e possíveis oportunidades de investigação.

### 7.2 Pergunta principal

**Como evoluíram as vendas da Olist ao longo do período analisado e quais diferenças podem ser observadas entre estados, valores dos pedidos e categorias de produtos?**

### 7.3 Metodologia

As conclusões foram elaboradas a partir de estatísticas descritivas, análises de distribuição, agrupamentos e visualizações gráficas.

Foram considerados 96.478 pedidos entregues, correspondentes a 110.197 registros de itens.

O valor total das vendas dos produtos, sem incluir fretes, foi de R$ 13.221.498,11.

Os procedimentos de limpeza e validação buscaram preservar a integridade dos dados, evitando exclusões ou alterações sem justificativa.

### 7.4 Organização das conclusões

As conclusões estão estruturadas em quatro dimensões:

1. Evolução temporal das vendas.
2. Distribuição geográfica das vendas.
3. Distribuição dos valores dos pedidos.
4. Participação das categorias de produtos.

Na sequência, serão apresentadas as limitações metodológicas e recomendações para análises futuras.



### 7.5 Principais Conclusões

A Análise Exploratória de Dados (EDA) permitiu identificar padrões relevantes no comportamento das vendas da Olist, considerando aspectos temporais, geográficos e comerciais.

#### 7.5.1 Evolução temporal das vendas

A análise mensal demonstrou crescimento expressivo no valor das vendas ao longo de 2017.

Em janeiro de 2017, o valor comercializado foi de R$ 111.798,36, enquanto novembro do mesmo ano registrou R$ 987.765,37, representando o maior valor mensal observado no período.

Durante 2018, entre janeiro e agosto, as vendas apresentaram oscilações, mantendo valores mensais superiores a R$ 800 mil.

Esses resultados indicam expansão do volume financeiro comercializado durante 2017, seguida de relativa estabilização em patamares mais elevados em 2018.

Entretanto, os dados não permitem afirmar quais fatores causaram essas variações.

#### 7.5.2 Distribuição geográfica das vendas

A análise por estado demonstrou concentração das vendas na região Sudeste.

Os três estados com maior valor comercializado foram:

- São Paulo: R$ 5.067.633,16 (38,33%).
- Rio de Janeiro: R$ 1.759.651,13 (13,31%).
- Minas Gerais: R$ 1.552.481,83 (11,74%).

Juntos, esses estados representaram aproximadamente 63,38% do valor total das vendas.

São Paulo destacou-se como o principal mercado consumidor da base analisada.

Os resultados demonstram diferenças relevantes na distribuição geográfica das vendas, mas não permitem estabelecer suas causas.

#### 7.5.3 Distribuição dos valores dos pedidos

A distribuição dos valores dos pedidos apresentou forte assimetria positiva.

Os principais indicadores foram:

- Média: R$ 137,04.
- Mediana: R$ 86,57.
- Desvio-padrão: R$ 209,05.
- Assimetria: 9,89.
- Valor máximo: R$ 13.440,00.

A diferença entre média e mediana demonstra a influência de pedidos com valores elevados sobre a média.

Pelo método do intervalo interquartil (IQR), foram identificados 7.659 possíveis outliers, correspondentes a 7,94% dos pedidos.

Esses registros foram preservados, pois valores elevados não representam necessariamente erros nos dados.

A mediana mostrou-se uma medida mais adequada para representar o valor central típico dos pedidos.

#### 7.5.4 Categorias com maior valor de vendas

A análise por categoria identificou diferenças na participação financeira dos grupos de produtos.

As cinco categorias com maior valor comercializado foram:

1. Beleza e saúde: R$ 1.233.131,72.
2. Relógios e presentes: R$ 1.166.176,98.
3. Cama, mesa e banho: R$ 1.023.434,76.
4. Esporte e lazer: R$ 954.852,55.
5. Informática e acessórios: R$ 888.724,61.

Essas cinco categorias representaram aproximadamente 39,83% do valor total das vendas.

A categoria cama, mesa e banho apresentou a maior quantidade de itens vendidos entre as cinco líderes, enquanto beleza e saúde apresentou o maior valor financeiro.

Esse resultado demonstra que quantidade de itens comercializados e valor total das vendas são indicadores diferentes.

#### 7.5.5 Síntese dos resultados

Os resultados evidenciaram quatro características principais:

1. Crescimento expressivo das vendas ao longo de 2017.
2. Concentração geográfica das vendas nos estados do Sudeste.
3. Distribuição assimétrica dos valores dos pedidos, com presença de valores extremos.
4. Participação financeira diferenciada entre as categorias de produtos.

Dessa forma, as análises permitiram responder à pergunta principal do projeto por meio de evidências quantitativas e visualizações.



### 7.6 Limitações da Análise

Embora os resultados tenham permitido identificar padrões relevantes, algumas limitações devem ser consideradas na interpretação dos dados.

#### 7.6.1 Período analisado

A análise temporal concentrou-se principalmente entre janeiro de 2017 e agosto de 2018.

Os meses com quantidade reduzida de registros foram mantidos na base, mas não utilizados no gráfico principal de evolução mensal, para evitar interpretações distorcidas.

#### 7.6.2 Seleção dos pedidos

Foram considerados apenas os pedidos com status de entrega concluída (`delivered`).

Essa escolha permitiu analisar transações efetivamente entregues, mas não representa todos os pedidos realizados na plataforma.

#### 7.6.3 Valores das vendas

O valor das vendas foi calculado a partir dos preços dos produtos, sem incluir os valores de frete.

Portanto, os resultados representam o valor comercializado dos produtos, e não necessariamente a receita líquida ou o lucro da empresa.

#### 7.6.4 Dados ausentes

Foram identificados valores ausentes em algumas variáveis.

As categorias de produtos ausentes foram identificadas como `categoria_nao_informada`, preservando os registros.

As datas ausentes foram mantidas sem preenchimento artificial, evitando a criação de informações inexistentes.

#### 7.6.5 Inconsistências cronológicas

Foram identificadas inconsistências entre datas de compra, envio e entrega.

Essas situações foram investigadas e documentadas durante o diagnóstico dos dados.

Os registros foram preservados, pois não havia evidências suficientes para determinar os valores corretos.

#### 7.6.6 Valores extremos

Os possíveis outliers identificados nos valores dos pedidos, preços e fretes foram mantidos.

Essa decisão considerou que valores extremos podem representar transações comerciais legítimas.

#### 7.6.7 Limitações interpretativas

A análise possui caráter exploratório e descritivo.

Os resultados permitem identificar padrões, diferenças e associações, mas não estabelecer relações de causa e efeito.



### 7.7 Recomendações e Trabalhos Futuros

Com base nos resultados obtidos, são propostas algumas possibilidades de aprofundamento das análises.

#### 7.7.1 Análise de sazonalidade

Investigar possíveis padrões sazonais nas vendas, considerando meses do ano, datas comemorativas e períodos promocionais.

Essa análise poderá ajudar a compreender as oscilações identificadas na evolução temporal.

#### 7.7.2 Análise regional

Investigar diferenças entre estados e regiões brasileiras, considerando quantidade de pedidos, ticket médio e participação financeira.

Essa abordagem poderá complementar a análise da concentração geográfica das vendas.

#### 7.7.3 Análise de categorias

Aprofundar o estudo das categorias de produtos, comparando quantidade de itens, preço médio e valor total comercializado.

O objetivo será identificar diferenças entre categorias com maior volume de vendas e categorias com maior valor financeiro.

#### 7.7.4 Análise logística

Investigar os prazos de entrega, atrasos e custos de frete, considerando as informações disponíveis na base.

Essa análise poderá contribuir para a compreensão de aspectos operacionais do comércio eletrônico.

#### 7.7.5 Análise de clientes

Utilizar o identificador único dos consumidores para investigar frequência de compras, recorrência e comportamento de consumo.

Essa abordagem poderá ampliar a compreensão do perfil dos consumidores da plataforma.

#### 7.7.6 Desenvolvimento de dashboards

Como continuidade do projeto, os indicadores poderão ser apresentados em dashboards interativos utilizando ferramentas como Power BI ou bibliotecas Python.

Essas visualizações poderão facilitar o acompanhamento dos resultados e a comparação entre períodos, estados e categorias.



### 7.8 Conclusão Final

O presente projeto teve como objetivo realizar uma Análise Exploratória de Dados (EDA) utilizando informações públicas do comércio eletrônico brasileiro disponibilizadas pela Olist.

O desenvolvimento foi organizado em etapas de compreensão dos dados, diagnóstico, limpeza, integração, validação, análise estatística e visualização.

Durante o tratamento dos dados, buscou-se preservar a integridade das informações, documentando os valores ausentes, possíveis outliers e inconsistências identificadas.

A análise considerou 96.478 pedidos entregues, totalizando R$ 13.221.498,11 em vendas de produtos, sem frete.

Os resultados demonstraram crescimento das vendas ao longo de 2017, concentração geográfica nos estados do Sudeste, assimetria positiva nos valores dos pedidos e diferenças na participação financeira das categorias de produtos.

As visualizações gráficas contribuíram para identificar e comunicar esses padrões de maneira objetiva.

Conclui-se que a aplicação de técnicas de programação em Python, manipulação de dados com Pandas e visualização de informações permite transformar dados brutos em informações relevantes para a compreensão do comportamento comercial.

O projeto também demonstrou a importância da qualidade dos dados, da documentação das decisões de tratamento e da validação dos resultados para garantir maior confiabilidade às análises.

Por fim, destaca-se que os resultados possuem caráter descritivo e podem servir como base para investigações futuras sobre sazonalidade, comportamento dos consumidores e desempenho comercial.

---

**Fim da Análise Exploratória de Dados — Projeto Olist**



---

## 8. Validação Final e Preparação para Entrega

### 8.1 Objetivo

Esta etapa tem como objetivo verificar a consistência dos resultados obtidos durante a Análise Exploratória de Dados (EDA) da Olist.

Serão realizadas verificações para confirmar:

- A quantidade de pedidos entregues analisados.
- A quantidade de itens após a integração das bases.
- A ausência de duplicidades nos identificadores dos pedidos agregados.
- A integridade dos valores totais de vendas.
- A consistência dos resultados por mês, estado e categoria.
- A existência das variáveis necessárias para as análises.

As verificações serão realizadas utilizando Python e Pandas, comparando os resultados calculados com os indicadores obtidos nas etapas anteriores.


In [ ]:

# VALIDAÇÃO FINAL DO PROJETO

import pandas as pd
import numpy as np

print("===== VALIDAÇÃO FINAL DO PROJETO OLIST =====")

# Valores esperados obtidos nas etapas anteriores
pedidos_esperados = 96478
itens_esperados = 110197
vendas_esperadas = 13221498.11

# Verificar quantidade de pedidos
teste_pedidos = (
    len(df_pedidos_valores) == pedidos_esperados
)

# Verificar quantidade de itens
teste_itens = (
    len(df_vendas_analise) == itens_esperados
)

# Verificar duplicidade de pedidos
teste_duplicados = (
    df_pedidos_valores["order_id"].is_unique
)

# Verificar total das vendas
total_vendas = df_pedidos_valores["valor_pedido"].sum()

teste_vendas = np.isclose(
    total_vendas,
    vendas_esperadas,
    atol=0.01
)

# Verificar valores por mês
teste_mensal = np.isclose(
    df_vendas_mensais["valor_vendas"].sum(),
    total_vendas,
    atol=0.01
)

# Verificar valores por estado
teste_estados = np.isclose(
    df_vendas_estados["valor_vendas"].sum(),
    total_vendas,
    atol=0.01
)

# Verificar valores por categoria
teste_categorias = np.isclose(
    df_vendas_categorias["valor_vendas"].sum(),
    total_vendas,
    atol=0.01
)

# Organizar resultados
validacoes = pd.DataFrame({
    "verificacao": [
        "Quantidade de pedidos",
        "Quantidade de itens",
        "Pedidos sem duplicidade",
        "Total das vendas",
        "Vendas mensais",
        "Vendas por estado",
        "Vendas por categoria"
    ],
    "resultado": [
        teste_pedidos,
        teste_itens,
        teste_duplicados,
        teste_vendas,
        teste_mensal,
        teste_estados,
        teste_categorias
    ]
})

display(validacoes)

print("\n===== RESULTADO FINAL =====")

if validacoes["resultado"].all():
    print("Todas as verificações foram aprovadas!")
else:
    print("Existem verificações que precisam de revisão.")



### 8.2 Resultados da Validação Final

A validação final do projeto foi realizada por meio de verificações automatizadas em Python, utilizando as bibliotecas Pandas e NumPy.

Foram executados sete testes de consistência, contemplando a quantidade de registros, a unicidade dos pedidos agregados e a integridade dos valores financeiros.

#### Resultados obtidos

| Verificação | Resultado |
|---|---|
| Quantidade de pedidos entregues | Aprovado |
| Quantidade de itens integrados | Aprovado |
| Ausência de duplicidade nos pedidos agregados | Aprovado |
| Consistência do valor total das vendas | Aprovado |
| Consistência das vendas mensais | Aprovado |
| Consistência das vendas por estado | Aprovado |
| Consistência das vendas por categoria | Aprovado |

**Resultado:** todas as sete verificações foram aprovadas.

Os resultados confirmam que os principais indicadores quantitativos e financeiros estão consistentes com os valores de referência definidos durante o projeto.

### 8.3 Considerações Finais da Validação

A execução dos testes permitiu verificar que as operações de integração, agregação e análise preservaram os principais totais esperados.

A validação contribui para a confiabilidade dos resultados apresentados, embora não elimine todas as limitações e inconsistências documentadas anteriormente.

Dessa forma, o projeto apresenta consistência interna nos indicadores avaliados e está preparado para a revisão final de execução e organização.

---
